## Preprocess Data the Male Training Data

In [ ]:
import pandas as pd

# File paths (updated for Extension_4.0)
base_path = '<LOCAL_PATH>'

# Define the datasets
datasets = {
    '(Train)': 'all_train_60_40.csv',
    '(Test)': 'all_test_60_40.csv'
}

# Define function to print unique IDs and status
def print_unique_ids_and_status(df, label):
    ids = sorted(df['id'].unique())
    print(f"\n📄 {label} — {len(ids)} unique IDs:")
    print(ids)

    # CWD Positive
    positive_ids = sorted(df.loc[df['CWD_status'] == 'positive', 'id'].unique())
    print(f"  ➡ CWD Positive IDs ({len(positive_ids)}): {positive_ids}")

    # CWD Negative
    negative_ids = sorted(df.loc[df['CWD_status'] == 'negative', 'id'].unique())
    print(f"  ➡ CWD Negative IDs ({len(negative_ids)}): {negative_ids}")

def run_datetime_eda(df, label):
    print(f"\n🧪 EDA on `datetime` for {label}")

    # Check for missing
    null_count = df['datetime'].isnull().sum()
    print(f"  ➤ Null values in `datetime`: {null_count}")

    # Convert to datetime
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    coercion_count = df['datetime'].isnull().sum() - null_count
    if coercion_count > 0:
        print(f"  ⚠️ {coercion_count} additional nulls after conversion (possibly bad formats)")

    # Check range
    min_time = df['datetime'].min()
    max_time = df['datetime'].max()
    print(f"  ➤ Overall date range: {min_time} to {max_time}")

    # Duration per ID
    id_group = df.groupby('id')['datetime']
    durations = (id_group.max() - id_group.min()).dt.total_seconds() / (60*60*24)
    print(f"  ➤ Duration (days) across IDs: min={durations.min():.2f}, max={durations.max():.2f}, median={durations.median():.2f}")

    # Continuity: time between rows per ID
    discontinuous_ids = []
    for _id, group in df.sort_values('datetime').groupby('id'):
        times = group['datetime'].dropna()
        deltas = times.diff().dt.total_seconds().dropna()
        if (deltas == 0).all():
            discontinuous_ids.append(_id)

    if discontinuous_ids:
        print(f"  ⚠️ {len(discontinuous_ids)} IDs have no time variation (i.e., all timestamps are identical):")
        print(f"     {discontinuous_ids[:10]}{' ...' if len(discontinuous_ids) > 10 else ''}")
    else:
        print(f"  ✅ All IDs have time variation.")

    return df

# Load and process each dataset
for label, file_name in datasets.items():
    path = f"{base_path}/{file_name}"
    df = pd.read_csv(path)
    print_unique_ids_and_status(df, label)
    df = run_datetime_eda(df, label)


unaffected_cwd_instances = [3, 4, 5, 7, 9, 13, 14, 22, 26, 27, 29, 33, 34, 38, 42, 43, 44, 45, 50, 51, 56, 60, 66, 76, 77, 78, 82, 84, 88, 89, 95, 99, 100, 104, 108, 109, 111, 115, 118, 119, 124, 129]

[1, 2, 6, 8, 11, 12, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 30, 31, 32, 35, 36, 37, 40, 47, 48, 53, 55, 57, 58, 61, 62, 64, 65, 67, 68, 69, 70, 71, 72, 73, 74, 79, 80, 83, 90, 91, 92, 93, 94, 96, 102, 105, 107, 113, 114, 116, 121, 122, 125, 130, 141]


5, 9, 14, 22, 26, 29, 44, 66, 95, 108, 118, 129

In [ ]:
import pandas as pd
import re

path = "<LOCAL_PATH>"

# Load raw CSV as strings to avoid coercion
df = pd.read_csv(path, dtype=str)

# Quick look at the first few
print("Sample pkeys:")
print(df["pkey"].head(10).tolist())

# Check unique formats using regex
pattern = re.compile(r"^\d+\.\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\d*$")

invalid = df.loc[~df["pkey"].str.match(pattern, na=False), "pkey"]

print(f"\nTotal rows: {len(df)}")
print(f"Invalid formatting count: {len(invalid)}")

if not invalid.empty:
    print("\nExamples of invalid pkeys:")
    print(invalid.sample(min(10, len(invalid))).tolist())


## Preprocessing Testing Data and Making Predictions on Anomalous Mule Deer

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Leak-free preprocessing with ID-disjoint splits and train-only fitting (with pkey)
# ──────────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler 

# ── REPRODUCIBILITY ───────────────────────────────────────────────────────────
os.environ["PYTHONHASHSEED"] = "0"
random.seed(13)
np.random.seed(13)

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base       = Path("<LOCAL_PATH>")
train_raw  = base/"all_train_60_40.csv"
test_raw   = base/"all_test_60_40.csv"

out_train  = base/"train_processed_60_40.csv"
out_val    = base/"val_processed_60_40.csv"
out_test   = base/"test_processed_60_40.csv"

enc_path   = base/"encoder_60_40.pkl"
sca_path   = base/"scaler_60_40.pkl"
feat_path  = base/"feature_columns_60_40.pkl"
split_path = base/"split_ids_60_40.pkl"

to_drop = [
    'x_2','x_1','x_2_2','transition_to_migratory','id_1','phases',
    'migration_class_grouped','hour','movement_group','r2n_smoothed',
    'suspect_jump','extreme_jump','SATS','rel.angle',
    'utm_easting','utm_northing','dx','dy','dt','cum_dist','dist_per_hr'
]

num_feats = [
    'velo','abs_rel_angle','sinuosity_28pt',
    'rolling_abs_rel_angle_mean','rolling_abs_rel_angle_sd',
    'rolling_abs_rel_angle_skew','rolling_abs_rel_angle_kurt',
    'rolling_velo_mean','rolling_velo_sd','rolling_velo_skew','rolling_velo_kurt',
    'low','est','high'
]

cat_feats  = ['season']
meta_feats = ['pkey','id','datetime']   # ← added pkey here

# ── UTILITIES ──────────────────────────────────────────────────────────────────
def basic_cleanup(df: pd.DataFrame) -> pd.DataFrame:
    """Drop unused cols, enforce numeric types, keep pkey intact, preserve order."""
    df = df.drop(columns=to_drop, errors='ignore').copy()
    
    # Preserve original datetime string AND a parsed version
    df['datetime_raw'] = df['datetime']   # keep raw untouched
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    
    for c in num_feats:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    
    df = df.dropna(subset=num_feats + cat_feats).copy()
    
    # Sort by id + parsed datetime (if valid), fallback to pkey for stability
    df = df.sort_values(['id', 'datetime', 'pkey']).reset_index(drop=True)
    return df

def transform_with_artifacts(df, encoder, scaler, feature_columns):
    """Apply fitted encoder/scaler; return DataFrame with pkey + meta + features."""
    cat_mat = encoder.transform(df[cat_feats])
    num_mat = scaler.transform(df[num_feats])
    cat_names = encoder.get_feature_names_out(cat_feats).tolist()
    X = np.hstack([num_mat, cat_mat])
    feat_df = pd.DataFrame(X, columns=num_feats + cat_names)

    feat_df = feat_df.reindex(columns=feature_columns, fill_value=0.0)
    out = pd.concat([df[meta_feats].reset_index(drop=True),
                     feat_df.reset_index(drop=True)], axis=1)
    return out

# ── 1) LOAD TRAIN POOL ────────────────────────────────────────────────────────
df_train_pool = pd.read_csv(train_raw)
df_train_pool = df_train_pool[df_train_pool['CWD_status'] == "negative"].copy()
df_train_pool = basic_cleanup(df_train_pool)

# ── 2) ID-DISJOINT SPLIT ──────────────────────────────────────────────────────
ids = df_train_pool['id'].unique()
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=13)
train_idx, val_idx = next(gss.split(df_train_pool, groups=df_train_pool['id']))

train_ids = df_train_pool.iloc[train_idx]['id'].unique()
val_ids   = df_train_pool.iloc[val_idx]['id'].unique()

df_train = df_train_pool[df_train_pool['id'].isin(train_ids)].copy()
df_val   = df_train_pool[df_train_pool['id'].isin(val_ids)].copy()

# ── 3) FIT ENCODER & SCALER ON TRAIN ONLY ─────────────────────────────────────
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
scaler  = MinMaxScaler()

encoder.fit(df_train[cat_feats])
scaler.fit(df_train[num_feats])

cat_names = encoder.get_feature_names_out(cat_feats).tolist()
feature_columns = num_feats + cat_names

# ── 4) TRANSFORM SPLITS ──────────────────────────────────────────────────────
df_train_out = transform_with_artifacts(df_train, encoder, scaler, feature_columns)
df_val_out   = transform_with_artifacts(df_val,   encoder, scaler, feature_columns)

# ── 5) TEST ───────────────────────────────────────────────────────────────────
df_test_raw = pd.read_csv(test_raw)
df_test_raw = basic_cleanup(df_test_raw)
df_test_out = transform_with_artifacts(df_test_raw, encoder, scaler, feature_columns)

# ── 6) SAVE ARTIFACTS ─────────────────────────────────────────────────────────
joblib.dump(encoder, enc_path)
joblib.dump(scaler,  sca_path)
joblib.dump(feature_columns, feat_path)
joblib.dump(
    {"train_ids": sorted(map(int, set(train_ids))),
     "val_ids":   sorted(map(int, set(val_ids))),
     "test_ids":  sorted(map(int, set(df_test_out['id'].unique())))},
    split_path
)

df_train_out.to_csv(out_train, index=False)
df_val_out.to_csv(out_val, index=False)
df_test_out.to_csv(out_test, index=False)

print("✅ Preprocessing complete with pkey preserved.")
print("Train IDs:", sorted(set(train_ids)))
print("Val IDs:",   sorted(set(val_ids)))
print("Test IDs:",  sorted(df_test_out['id'].unique().tolist()))


In [ ]:
# ── Quick EDA on processed splits ─────────────────────────────────────────────
import pandas as pd

def quick_eda(path, name):
    df = pd.read_csv(path, parse_dates=["datetime"], infer_datetime_format=True)
    
    print(f"\n📊 {name} Split")
    print(f"  → Rows: {len(df)}")
    print(f"  → Unique IDs: {df['id'].nunique()}")
    
    # temporal range
    min_dt, max_dt = df['datetime'].min(), df['datetime'].max()
    print(f"  → Temporal range: {min_dt} → {max_dt}")
    
    # per-ID summary
    id_summary = df.groupby("id")["datetime"].agg(["min", "max", "count"])
    print("  → Per-ID counts (first 5 rows):")
    print(id_summary.head())
    return id_summary

train_summary = quick_eda(out_train, "Train")
val_summary   = quick_eda(out_val, "Validation")
test_summary  = quick_eda(out_test, "Test")

# Optional: Save summaries for manual inspection
train_summary.to_csv(base/"train_id_summary.csv")
val_summary.to_csv(base/"val_id_summary.csv")
test_summary.to_csv(base/"test_id_summary.csv")


## EDA of clean up

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

# === parser identical to the rescue logic ===
_TIME_SPLIT_RE = re.compile(r'^(\d{4}-\d{2}-\d{2})[ T](\d{2}):(\d{2}):(\d+)$')

def _parse_pkey_timestamp(s: str) -> pd.Timestamp:
    if not isinstance(s, str):
        return pd.NaT
    s = s.strip().replace('T', ' ')
    m = _TIME_SPLIT_RE.match(s)
    if not m:
        return pd.NaT
    date_part, hh, mm, sec_ms = m.groups()
    try:
        hh_i = int(hh); mm_i = int(mm)
    except ValueError:
        return pd.NaT

    if len(sec_ms) <= 2:
        sec_i = int(sec_ms); ms_i = 0
    else:
        sec_i = int(sec_ms[:2])
        ms_i  = int(sec_ms[2:][:3].ljust(3, '0'))  # up to 3 digits as ms

    base_date = pd.to_datetime(date_part, errors='coerce')
    if pd.isna(base_date):
        return pd.NaT
    return base_date + pd.to_timedelta(f"{hh_i:02d}:{mm_i:02d}:{sec_i:02d}.{ms_i:03d}")

def eda_date_vs_pkey(path: Path, name: str, id_col="id", dt_col="datetime", pkey_col="pkey", tol_ms=1.0):
    print(f"\n📊 {name} — {path.name}")
    df = pd.read_csv(path)
    # robust parse of datetime column
    df[dt_col] = pd.to_datetime(df[dt_col], errors='coerce')
    n = len(df)

    # parse timestamp from pkey tail "<id>.<YYYY-MM-DD HH:MM:SSmmm...>"
    if pkey_col not in df.columns:
        print("  ❌ No pkey column present; skipping.")
        return

    pkey_tail = df[pkey_col].astype(str).str.split('.', n=1).str[1]
    ts_from_pkey = pkey_tail.apply(_parse_pkey_timestamp)

    # basic availability
    n_dt_nat   = int(df[dt_col].isna().sum())
    n_pkey_nat = int(ts_from_pkey.isna().sum())
    print(f"  Rows: {n:,} | NaT(datetime)={n_dt_nat:,} | NaT(pkey_ts)={n_pkey_nat:,}")

    # compare (only where both are present)
    mask_ok = df[dt_col].notna() & ts_from_pkey.notna()
    delta = (df.loc[mask_ok, dt_col] - ts_from_pkey.loc[mask_ok]).dt.total_seconds() * 1000.0  # ms
    n_cmp = int(mask_ok.sum())
    if n_cmp == 0:
        print("  ⚠️ No rows with both datetime and pkey-derived timestamps.")
        return

    # exact/near-exact match stats
    tol = float(tol_ms)
    n_exact = int((delta == 0).sum())
    n_close = int((delta.abs() <= tol).sum())
    print(f"  Compareable rows: {n_cmp:,}")
    print(f"   • exact matches (Δ=0 ms): {n_exact:,} ({n_exact/n_cmp:.1%})")
    print(f"   • near matches  (|Δ| ≤ {tol_ms} ms): {n_close:,} ({n_close/n_cmp:.1%})")
    if n_cmp - n_close > 0:
        print(f"   • mismatches    (|Δ| > {tol_ms} ms): {n_cmp - n_close:,} ({(n_cmp-n_close)/n_cmp:.1%})")
        print(f"     Δ(ms): min={delta.min():.3f} | p1={np.percentile(delta,1):.3f} | "
              f"median={np.median(delta):.3f} | p99={np.percentile(delta,99):.3f} | max={delta.max():.3f}")

        # show a few mismatches
        bad_idx = delta[delta.abs() > tol].sort_values().index
        sample = df.loc[bad_idx, [id_col, pkey_col, dt_col]].copy()
        sample["pkey_ts"] = ts_from_pkey.loc[bad_idx]
        sample["delta_ms"] = delta.loc[bad_idx]
        print("   • sample mismatches:")
        print(sample.head(5).to_string(index=False))

    # duplicates
    dup_pkey = int(df[pkey_col].duplicated().sum()) if pkey_col in df else 0
    dup_id_dt = int(df[[id_col, dt_col]].duplicated().sum())
    print(f"  Duplicates: pkey={dup_pkey:,} | (id, datetime)={dup_id_dt:,}")

    # per-ID monotonicity (any backward time jump?)
    gb = df.sort_values([id_col, dt_col, pkey_col]).groupby(id_col, sort=False)
    has_backwards = gb[dt_col].apply(lambda s: (s.diff() < pd.Timedelta(0)).any())
    n_back_ids = int(has_backwards.sum())
    print(f"  IDs with non-monotonic datetime: {n_back_ids:,} "
          f"out of {df[id_col].nunique():,}")
    if n_back_ids:
        print("   • example IDs with backwards jumps:", has_backwards[has_backwards].head(5).index.tolist())

    # ordering consistency between datetime and pkey_ts
    # (where both available, by ID)
    def _order_inversions(sub):
        sub = sub[[dt_col]].copy()
        sub["pkey_ts"] = ts_from_pkey.loc[sub.index]
        sub = sub.dropna()
        if len(sub) <= 1:
            return 0
        # number of pairwise inversions between the two orderings
        # proxy: count sign mismatches of successive diffs
        dt_diff   = sub[dt_col].diff().dt.total_seconds()
        pkey_diff = sub["pkey_ts"].diff().dt.total_seconds()
        return int(((np.sign(dt_diff.fillna(0)) != np.sign(pkey_diff.fillna(0))) & (dt_diff != 0) & (pkey_diff != 0)).sum())

    inv_counts = gb.apply(_order_inversions)
    n_ids_with_inversions = int((inv_counts > 0).sum())
    if n_ids_with_inversions:
        print(f"  IDs with ordering inconsistencies (datetime vs pkey_ts): {n_ids_with_inversions:,}")
        print("   • example:", inv_counts[inv_counts > 0].head(5).to_dict())
    else:
        print("  ✅ datetime ordering matches pkey_ts ordering for all IDs checked.")

# ===== run on your three outputs =====
base = Path("<LOCAL_PATH>")
eda_date_vs_pkey(base/"train_processed_60_40.csv", "Train")
eda_date_vs_pkey(base/"val_processed_60_40.csv",   "Val")
eda_date_vs_pkey(base/"test_processed_60_40.csv",  "Test")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Autoencoder training USING CUSTOM LOSS + flexible thresholding
# ──────────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ===== Reproducibility =====
os.environ["PYTHONHASHSEED"] = "0"
random.seed(13); np.random.seed(13)
tf.keras.utils.set_random_seed(13)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# ===== Config / Paths =====
base = "<LOCAL_PATH>"
train_path = f"{base}/train_processed_60_40.csv"
val_path   = f"{base}/val_processed_60_40.csv"   # negatives-only val (from preprocessing)
test_path  = f"{base}/test_processed_60_40.csv"

feat_cols_path = f"{base}/feature_columns_60_40.pkl"
model_path     = f"{base}/autoencoder_model.h5"
threshold_path = f"{base}/reconstruction_threshold.pkl"
history_path   = f"{base}/training_history.npy"
ckpt_path      = f"{base}/ae_ckpt.weights.h5"

# ===== Threshold policy (choose one) =====
THRESHOLD_POLICY = "train_lowest_p95"   # "train_lowest_p95" | "train_quantile" | "val_row_fpr" | "val_id_flag_rate" | "mad"

# Params for each policy
LOWEST_FRAC   = 0.75      # for "train_lowest_p95" 
LOWEST_Q      = 90        # percentile on the lowest fraction
TRAIN_Q       = 0.90      # for "train_quantile": e.g., 90th percentile of all train errors
VAL_ROW_FPR   = 0.20      # for "val_row_fpr": allow 20% of NEG val rows to flag → lower θ, more flags
VAL_ID_RATE   = 0.20      # for "val_id_flag_rate": allow 20% of NEG val IDs to flag at least 1 row
MIN_ROWS_PER_ID = 1       # ignore super-short IDs when computing ID-level rate
MAD_K         = 3.0       # for "mad"


# ===== Load features with frozen column order =====
feature_columns = joblib.load(feat_cols_path)

def load_X(path):
    df = pd.read_csv(path)
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    X = df[feature_columns].values.astype(np.float32)
    meta = df[["id"]].copy()
    if "datetime" in df.columns:
        meta["datetime"] = df["datetime"].values
    return X, meta, df

X_train, meta_train, df_train_raw = load_X(train_path)
X_val,   meta_val,   df_val_raw   = load_X(val_path)
X_test,  meta_test,  df_test_raw  = load_X(test_path)

# ===== tf.data pipelines =====
batch_size = 128
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, X))
    if training:
        ds = ds.shuffle(min(10000, len(X)), seed=13, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False).cache().prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(X_train, training=True)
val_ds   = make_ds(X_val,   training=False)

# ===== Model =====
input_dim = X_train.shape[1]
reg = regularizers.l2(1e-4)

inp = layers.Input(shape=(input_dim,))
x = layers.Dense(32, activation='relu', kernel_regularizer=reg)(inp)
x = layers.Dense(16, activation='relu', kernel_regularizer=reg)(x)
z = layers.Dense(4,  activation='relu', kernel_regularizer=reg, name="bottleneck")(x)
y = layers.Dense(16, activation='relu', kernel_regularizer=reg)(z)
y = layers.Dense(32, activation='relu', kernel_regularizer=reg)(y)
out = layers.Dense(input_dim, activation='linear')(y)
autoencoder = Model(inp, out, name="autoencoder")

# === Custom loss ===
LAMBDA_DEV = 0.5                      # ← your choice
LOSS_SCALE = 1.0 + LAMBDA_DEV*input_dim  # α = 1 + λ d

def get_custom_loss(loss_scale=LOSS_SCALE):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)  # mean over features
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(LOSS_SCALE)
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=loss_fn)

# ===== Train =====
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
ckpt = ModelCheckpoint(filepath=ckpt_path, save_weights_only=True, monitor="val_loss", save_best_only=True)

history = autoencoder.fit(
    train_ds, epochs=100, validation_data=val_ds, callbacks=[early_stop, ckpt], verbose=1
)

autoencoder.save(model_path)
np.save(history_path, history.history, allow_pickle=True)
print(f"✅ Saved model → {model_path}")

# ===== Errors on train/val (row-level, using SAME loss) =====
recon_train = autoencoder.predict(X_train, batch_size=batch_size, verbose=0)
train_errors = loss_fn(X_train, recon_train).numpy().astype(np.float64)

recon_val = autoencoder.predict(X_val, batch_size=batch_size, verbose=0)
val_errors = loss_fn(X_val, recon_val).numpy().astype(np.float64)

# ===== Threshold policies =====
def choose_threshold(policy: str):
    info = {"policy": policy}   # <— standard key

    if policy == "train_lowest_p95":
        n_low = int(len(train_errors) * LOWEST_FRAC)
        base = np.sort(train_errors)[:n_low]
        theta = float(np.percentile(base, LOWEST_Q))
        info.update({"lowest_frac": LOWEST_FRAC, "percentile": LOWEST_Q, "n_low": n_low})

    elif policy == "train_quantile":
        theta = float(np.quantile(train_errors, TRAIN_Q))
        info.update({"train_quantile": TRAIN_Q})

    elif policy == "val_row_fpr":
        q = 1.0 - float(VAL_ROW_FPR)
        theta = float(np.quantile(val_errors, q))
        info.update({"val_row_fpr": VAL_ROW_FPR, "val_quantile": q})

    elif policy == "val_id_flag_rate":
        q_grid = np.linspace(0.50, 0.999, 200)
        ids = df_val_raw["id"].values
        best_theta, best_rate, best_q = None, None, None
        for q in q_grid:
            th = np.quantile(val_errors, q)
            row_flag = val_errors > th
            id_any = pd.Series(row_flag).groupby(ids, sort=False).any()
            n_rows = pd.Series(1, index=ids).groupby(ids).sum()
            eligible = n_rows[n_rows >= MIN_ROWS_PER_ID].index
            rate = float(id_any.reindex(eligible).fillna(False).mean())
            if rate <= VAL_ID_RATE:
                best_theta, best_rate, best_q = float(th), rate, float(q)
                break
        if best_theta is None:
            best_q = float(q_grid[-1])
            best_theta = float(np.quantile(val_errors, best_q))
            row_flag = val_errors > best_theta
            id_any = pd.Series(row_flag).groupby(ids, sort=False).any()
            n_rows = pd.Series(1, index=ids).groupby(ids).sum()
            eligible = n_rows[n_rows >= MIN_ROWS_PER_ID].index
            best_rate = float(id_any.reindex(eligible).fillna(False).mean())
        theta = best_theta
        info.update({"val_id_rate_target": VAL_ID_RATE, "achieved_rate": best_rate, "val_quantile": best_q})

    elif policy == "mad":
        med = float(np.median(train_errors))
        mad = float(np.median(np.abs(train_errors - med)) + 1e-12)
        theta = med + MAD_K * 1.4826 * mad
        info.update({"median": med, "mad": mad, "k": MAD_K})

    else:
        raise ValueError(f"Unknown THRESHOLD_POLICY: {policy}")

    return theta, info

theta, th_info = choose_threshold(THRESHOLD_POLICY)

# Persist policy hyperparameters for reproducibility
if THRESHOLD_POLICY == "train_lowest_p95":
    th_info.update({"LOWEST_FRAC": float(LOWEST_FRAC), "LOWEST_Q": float(LOWEST_Q)})
elif THRESHOLD_POLICY == "train_quantile":
    th_info.update({"TRAIN_Q": float(TRAIN_Q)})
elif THRESHOLD_POLICY == "val_row_fpr":
    th_info.update({"VAL_ROW_FPR": float(VAL_ROW_FPR)})
elif THRESHOLD_POLICY == "val_id_flag_rate":
    th_info.update({"VAL_ID_RATE": float(VAL_ID_RATE), "MIN_ROWS_PER_ID": int(MIN_ROWS_PER_ID)})
elif THRESHOLD_POLICY == "mad":
    th_info.update({"MAD_K": float(MAD_K)})

th_info["effective_multiplier"] = float(LOSS_SCALE)  # alias for reviewers
print(f"[loss] λ={LAMBDA_DEV} | d={input_dim} | α=1+λ·d={LOSS_SCALE:.3f} (scaled-MSE)")


# ===== Diagnostics: how aggressive is θ? =====
def percent_rows_flagged(errors, theta):
    return float(np.mean(errors > theta))

train_row_rate = percent_rows_flagged(train_errors, theta)
val_row_rate   = percent_rows_flagged(val_errors, theta)

# ID-level on val (negatives)
val_ids = df_val_raw["id"].values
val_row_flag = val_errors > theta
val_id_any = pd.Series(val_row_flag).groupby(val_ids, sort=False).any()
val_n_rows = pd.Series(1, index=val_ids).groupby(val_ids).sum()
val_eligible = val_n_rows[val_n_rows >= MIN_ROWS_PER_ID].index
val_id_rate = float(val_id_any.reindex(val_eligible).fillna(False).mean())

print(f"📏 θ = {theta:.6g} | policy = {THRESHOLD_POLICY}")
print(f"   Train rows flagged: {100*train_row_rate:.1f}%")
print(f"   Val   rows flagged: {100*val_row_rate:.1f}%")
print(f"   Val     IDs flagged (≥1 row): {100*val_id_rate:.1f}% (min_rows={MIN_ROWS_PER_ID})")

# ===== Persist threshold + bookkeeping =====
# Diagnostics
# (unchanged, but print 'policy')
print(f"📏 θ = {theta:.6g} | policy = {THRESHOLD_POLICY}")

# Persist threshold + bookkeeping (use 'policy')
th_info.update({
    "policy": THRESHOLD_POLICY,
    "theta": float(theta),
    "lambda_dev": float(LAMBDA_DEV),
    "input_dim": int(input_dim),
    "loss_scale": float(LOSS_SCALE),  # α
    "loss_name": "scaled_mse_equivalent_to_mse_plus_lambda_sse",
    "train_rows_flagged": train_row_rate,
    "val_rows_flagged": val_row_rate,
    "val_ids_flagged": val_id_rate,
    "min_rows_per_id": int(MIN_ROWS_PER_ID)
})
joblib.dump(th_info, threshold_path)
print(f"✅ Threshold saved → {threshold_path}")


In [ ]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
val_csv         = base / "val_processed_60_40.csv"
model_path      = base / "autoencoder_model.h5"
threshold_path  = base / "reconstruction_threshold.pkl"   # contains θ + loss params from training
feat_cols_path  = base / "feature_columns_60_40.pkl"      # enforce column order
param_out_path  = base / "id_rule_params.pkl"

USE_MIXED_VAL = False  # your val is negatives-only per preprocessing

# preregistered operating point (choose & freeze before test)
FROZEN_COUNT     = 75
FROZEN_FRAC      = 0.75
MIN_ROWS_PER_ID  = 1          # consider 100 if short IDs are noisy
AGGREGATION_RULE = "OR"       # freeze: "OR", "AND", "FRAC_ONLY", or "COUNT_ONLY"

# if you ever use mixed val with labels, these grids are considered a priori
COUNTS = [50, 100, 150, 200]
FRACS  = [0.80, 0.90, 0.925]
POS_IDS_VAL = []   # only used when USE_MIXED_VAL=True

# ── LOAD MODEL / FEATURES / TRAIN ARTIFACT ────────────────────────────────────
feature_columns = joblib.load(feat_cols_path)
autoencoder = load_model(model_path, compile=False)

# load training artifact with θ + loss params
th_info   = joblib.load(threshold_path)
theta     = float(th_info["theta"])
policy    = th_info.get("policy", "unknown")

# reconstruct scaled-MSE loss consistently
lambda_dev = float(th_info.get("lambda_dev", 0.5))
input_dim  = int(th_info.get("input_dim", len(feature_columns)))
loss_scale = float(th_info.get("loss_scale", 1.0 + lambda_dev * input_dim))

def get_custom_loss(loss_scale=loss_scale):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(loss_scale)
print(f"Using TRAIN-derived θ = {theta:.6f} (policy={policy}) | λ={lambda_dev} | d={input_dim} | α={loss_scale:.6g}")

# ── LOAD VAL AND COMPUTE ROW ERRORS (FROZEN FEATURE ORDER) ────────────────────
df_val = pd.read_csv(val_csv)
if "datetime" in df_val.columns:
    df_val["datetime"] = pd.to_datetime(df_val["datetime"], errors="coerce")

X_val     = df_val[feature_columns].values.astype(np.float32)
recon_val = autoencoder.predict(X_val, batch_size=128, verbose=0)
val_err   = loss_fn(X_val, recon_val).numpy().astype(np.float64)

rows_val = pd.DataFrame({
    "id": df_val["id"].values,
    "datetime": df_val["datetime"] if "datetime" in df_val.columns else pd.NaT,
    "reconstruction_error": val_err
})
rows_val["row_flag"] = rows_val["reconstruction_error"] > theta

# ── AGGREGATE BY ID ───────────────────────────────────────────────────────────
g         = rows_val.groupby("id", sort=False)
n_rows    = g.size()
n_flags   = g["row_flag"].sum()
flag_frac = n_flags / n_rows

# consistent eligible set (evaluation/selection must use the same gate)
eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
n_rows    = n_rows.reindex(eligible_idx).astype(int)
n_flags   = n_flags.reindex(eligible_idx).astype(int)
flag_frac = flag_frac.reindex(eligible_idx).astype(float)

# ── SELECT (COUNT, FRAC) IF MIXED VAL; OTHERWISE FREEZE A PRIORI ─────────────
if USE_MIXED_VAL:
    pos_val = set(POS_IDS_VAL) & set(eligible_idx)
    if len(pos_val) == 0:
        raise ValueError("No positive IDs present in eligible validation set.")

    best = None; best_key = None
    for c in COUNTS:
        for f in FRACS:
            idx_count = set(n_flags[n_flags >= c].index)
            idx_frac  = set(flag_frac[flag_frac >= f].index)

            if   AGGREGATION_RULE == "AND":       pred = idx_count & idx_frac
            elif AGGREGATION_RULE == "FRAC_ONLY": pred = idx_frac
            elif AGGREGATION_RULE == "COUNT_ONLY":pred = idx_count
            else:                                 pred = idx_count | idx_frac  # OR

            recall = len(pred & pos_val) / len(pos_val)
            key = (recall, c, f)  # maximize recall; tie-break to stricter (higher c,f)
            if (best is None) or (key > best_key):
                best = {"count": int(c), "frac": float(f), "val_recall": float(recall)}
                best_key = key

    COUNT, FRAC = best["count"], best["frac"]
    print(f"Frozen ID rule from validation (positives-only, {AGGREGATION_RULE}): "
          f"COUNT={COUNT} | FRAC={FRAC} | recall={best['val_recall']:.3f}")
else:
    COUNT, FRAC = int(FROZEN_COUNT), float(FROZEN_FRAC)
    print(f"Using pre-registered ID rule ({AGGREGATION_RULE}; negatives-only val): "
          f"COUNT={COUNT} | FRAC={FRAC} | min_rows={MIN_ROWS_PER_ID}")

# ── SAVE FROZEN PARAMS FOR TEST ───────────────────────────────────────────────
frozen = {
    "theta": float(theta),
    "count": int(COUNT),
    "frac": float(FRAC),
    "lambda_dev": float(lambda_dev),
    "input_dim": int(input_dim),
    "loss_scale": float(loss_scale),
    "policy": policy,
    "aggregation_rule": AGGREGATION_RULE,  # ← frozen & explicit
    "val_meta": {
        "min_rows_per_id": int(MIN_ROWS_PER_ID),
        "use_mixed_val": bool(USE_MIXED_VAL)
    }
}
joblib.dump(frozen, param_out_path)
print(f"✅ Saved frozen params to: {param_out_path}")


In [ ]:

rows_val_out = str(val_csv).replace(
    ".csv",
    "_rows_with_flags_FROZEN.csv"
)

rows_val["row_flag"] = rows_val["reconstruction_error"] > theta

rows_val.to_csv(rows_val_out, index=False)
print(f"✅ Saved validation row flags → {rows_val_out}")

In [ ]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score, precision_recall_fscore_support
from pathlib import Path

# ── PATHS ──────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
test_feat_path  = base / "test_processed_60_40.csv"
model_path      = base / "autoencoder_model.h5"
frozen_param_p  = base / "id_rule_params.pkl"           # produced in validation (frozen params)
feat_cols_path  = base / "feature_columns_60_40.pkl"    # frozen feature order

rows_out_csv    = str(test_feat_path).replace(".csv", "_rows_with_flags_FROZEN.csv")
ids_out_csv     = str(test_feat_path).replace(".csv", "_id_summary_FROZEN.csv")

# (Optional) test labels for evaluation
actual_cwd_instances_test     = [1, 2, 6, 8, 11, 12, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 30, 31, 32, 35, 36, 37, 40, 47, 48, 53, 57, 58, 61, 62, 64, 65, 67, 68, 69, 71, 72, 74, 79, 83, 90, 91, 92, 93, 94, 96, 102, 105, 113, 114, 121, 122, 125, 130, 141]
unaffected_cwd_instances_test = [5, 9, 14, 22, 26, 29, 44, 55, 66, 95, 108, 118, 129]

# ── LOAD ARTIFACTS ────────────────────────────────────────────────────────────
feature_columns = joblib.load(feat_cols_path)           # list[str]
frozen = joblib.load(frozen_param_p)                    # dict from validation

theta  = float(frozen["theta"])
COUNT  = int(frozen["count"])
FRAC   = float(frozen["frac"])
policy = frozen.get("policy", "unknown")

# loss consistency (scaled-MSE, α = 1 + λ·d)
lambda_dev = float(frozen.get("lambda_dev", 0.5))
input_dim  = int(frozen.get("input_dim", len(feature_columns)))
loss_scale = float(frozen.get("loss_scale", 1.0 + lambda_dev * input_dim))

# aggregation rule + rows gate
AGGREGATION_RULE = str(frozen.get("aggregation_rule", "OR")).upper()  # "OR" | "AND" | "FRAC_ONLY" | "COUNT_ONLY"
MIN_ROWS_PER_ID  = int(frozen.get("val_meta", {}).get("min_rows_per_id", 1))

print(f"Using frozen params → θ={theta:.6f} | COUNT={COUNT} | FRAC={FRAC} | "
      f"λ={lambda_dev} | d={input_dim} | α={loss_scale:.3f} | MIN_ROWS={MIN_ROWS_PER_ID} | "
      f"policy={policy} | agg_rule={AGGREGATION_RULE}")

def get_custom_loss(loss_scale=loss_scale):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)  # mean over features
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(loss_scale)
autoencoder = load_model(model_path, compile=False)

# ── LOAD TEST AND COMPUTE ROW ERRORS (WITH FROZEN FEATURE ORDER) ──────────────
df = pd.read_csv(test_feat_path)
if "datetime" in df.columns:
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

X_test = df[feature_columns].values.astype(np.float32)
recon  = autoencoder.predict(X_test, batch_size=128, verbose=0)
row_err = loss_fn(X_test, recon).numpy().astype(np.float64)

rows = pd.DataFrame({
    "id": df["id"].values,
    "datetime": df["datetime"] if "datetime" in df.columns else pd.NaT,
    "reconstruction_error": row_err
})
# Note: '>' is the frozen policy; use '≥' only if you state that in Methods.
rows["row_flag"] = rows["reconstruction_error"] > theta

# ── AGGREGATE BY ID USING FROZEN RULE ─────────────────────────────────────────
g        = rows.groupby("id", sort=False)
n_rows   = g.size()
n_flags  = g["row_flag"].sum()
flag_fr  = n_flags / n_rows
id_max   = g["reconstruction_error"].max()
id_mean  = g["reconstruction_error"].mean()

eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
n_flags_elig = n_flags.reindex(eligible_idx)
flag_fr_elig = flag_fr.reindex(eligible_idx)

# Predictions per frozen aggregation rule
if AGGREGATION_RULE == "AND":
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index) & set(flag_fr_elig[flag_fr_elig >= FRAC].index)
elif AGGREGATION_RULE == "FRAC_ONLY":
    pred_ids = set(flag_fr_elig[flag_fr_elig >= FRAC].index)
elif AGGREGATION_RULE == "COUNT_ONLY":
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index)
else:  # "OR" (guard: COUNT must be > 0 or OR degenerates)
    if COUNT <= 0:
        raise ValueError("AGGREGATION_RULE='OR' requires COUNT>0; otherwise the count branch trivializes.")
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index) | set(flag_fr_elig[flag_fr_elig >= FRAC].index)

pred_ids = sorted(pred_ids)
print("Predicted positive IDs (frozen rule):", pred_ids)

# ── SAVE ARTIFACTS ────────────────────────────────────────────────────────────
rows.to_csv(rows_out_csv, index=False)

# Rule-aligned and component scores (for plots & AUC)
eps = 1e-12
count_norm_all = n_flags / max(COUNT, eps)
frac_norm_all  = flag_fr / max(FRAC, eps)

if AGGREGATION_RULE == "AND":
    score_all = np.minimum(count_norm_all, frac_norm_all)
elif AGGREGATION_RULE == "FRAC_ONLY":
    score_all = frac_norm_all
elif AGGREGATION_RULE == "COUNT_ONLY":
    score_all = count_norm_all
else:  # OR
    score_all = np.maximum(count_norm_all, frac_norm_all)

ids_df = pd.DataFrame({
    "id": n_rows.index,
    "n_rows": n_rows.values,
    "n_flags": n_flags.reindex(n_rows.index).values,
    "flag_frac": flag_fr.reindex(n_rows.index).values,
    "rule_aligned_score": score_all.reindex(n_rows.index).values,
    "frac_norm_score": frac_norm_all.reindex(n_rows.index).values,
    "count_norm_score": count_norm_all.reindex(n_rows.index).values,
    "id_max_error": id_max.reindex(n_rows.index).values,
    "id_mean_error": id_mean.reindex(n_rows.index).values,
    "eligible": [1 if i in set(eligible_idx) else 0 for i in n_rows.index],
    "is_positive_id": [1 if i in pred_ids else 0 for i in n_rows.index]
})
ids_df.to_csv(ids_out_csv, index=False)
print(f"✅ Wrote: {rows_out_csv}\n✅ Wrote: {ids_out_csv}")

# ── (OPTIONAL) TEST EVALUATION — eligible set only, rule-aligned score ────────
pos_set = set(actual_cwd_instances_test)
neg_set = set(unaffected_cwd_instances_test)

eval_ids   = sorted((pos_set | neg_set) & set(eligible_idx))
n_pos_eval = sum(i in pos_set for i in eval_ids)
n_neg_eval = sum(i in neg_set for i in eval_ids)
print(f"Eval on eligible IDs only → n_eval={len(eval_ids)} (pos={n_pos_eval}, neg={n_neg_eval}; min_rows={MIN_ROWS_PER_ID})")

if eval_ids and (n_pos_eval > 0 or n_neg_eval > 0):
    y_true = np.array([1 if i in pos_set else 0 for i in eval_ids])
    y_pred = np.array([1 if i in pred_ids else 0 for i in eval_ids])

    if (y_true.min() != y_true.max()):
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        TN, FP, FN, TP = cm.ravel()

        # Required metrics
        sensitivity = TP / (TP + FN) if (TP + FN) else 0.0                   # True Positive Rate
        specificity = TN / (TN + FP) if (TN + FP) else 0.0                   # True Negative Rate
        precision   = TP / (TP + FP) if (TP + FP) else 0.0
        accuracy    = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) else 0.0
        f1_score    = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) else 0.0
        tss         = sensitivity + specificity - 1
        prevalence  = (TP + FN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) else 0.0
        bal_acc     = 0.5 * (sensitivity + specificity)

        # Print full metrics block (match your paragraph exactly)
        print("\n=== Test Metrics (Frozen Rule, Eligible IDs) ===")
        print(f"TP = {TP}, FP = {FP}, FN = {FN}, TN = {TN}")
        print(f"Sensitivity     (TPR) = {sensitivity:.3f}")
        print(f"Specificity     (TNR) = {specificity:.3f}")
        print(f"Precision              = {precision:.3f}")
        print(f"Accuracy               = {accuracy:.3f}")
        print(f"F1 Score               = {f1_score:.3f}")
        print(f"True Skill Statistic   = {tss:.3f}")
        print(f"Prevalence Identified  = {prevalence:.3f}")
        print(f"Balanced Accuracy      = {bal_acc:.3f}")

        def wilson_ci(k, n, z=1.96):
            if n == 0: return (np.nan, np.nan)
            p = k / n
            denom = 1 + z*z/n
            center = (p + z*z/(2*n)) / denom
            half = z * np.sqrt((p*(1-p)/n) + (z*z/(4*n*n))) / denom
            return (center - half, center + half)

        prec_lo, prec_hi = wilson_ci(TP, TP + FP)
        sens_lo, sens_hi = wilson_ci(TP, TP + FN)
        spec_lo, spec_hi = wilson_ci(TN, TN + FP)
        tss_lo, tss_hi   = sens_lo + spec_lo - 1, sens_hi + spec_hi - 1

        print("\n95% Confidence Intervals:")
        print(f"  Precision     = [{prec_lo:.3f}, {prec_hi:.3f}]")
        print(f"  Sensitivity   = [{sens_lo:.3f}, {sens_hi:.3f}]")
        print(f"  Specificity   = [{spec_lo:.3f}, {spec_hi:.3f}]")
        print(f"  TSS           = [{tss_lo:.3f}, {tss_hi:.3f}]")



Using frozen params → θ=0.061986 | COUNT=75 | FRAC=0.75 | λ=0.5 | d=16 | α=9.000 | MIN_ROWS=1 | policy=train_lowest_p95 | agg_rule=OR
2026-07-20 12:34:18.287069: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-07-20 12:34:18.287127: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-07-20 12:34:18.287140: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1784576058.287200 2600515 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1784576058.287227 2600515 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-07-20 12:34:19.041811: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
Predicted positive IDs (frozen rule): [1, 2, 5, 6, 8, 11, 12, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 29, 30, 31, 35, 36, 37, 40, 47, 53, 57, 58, 61, 62, 64, 65, 66, 67, 68, 69, 70, 71, 72, 74, 79, 83, 90, 91, 92, 93, 94, 95, 96, 102, 105, 107, 113, 114, 116, 121, 130, 141]
✅ Wrote: <LOCAL_PATH>
✅ Wrote: <LOCAL_PATH>
Eval on eligible IDs only → n_eval=68 (pos=55, neg=13; min_rows=1)

=== Test Metrics (Frozen Rule, Eligible IDs) ===
TP = 50, FP = 6, FN = 5, TN = 7
Sensitivity     (TPR) = 0.909
Specificity     (TNR) = 0.538
Precision              = 0.893
Accuracy               = 0.838
F1 Score               = 0.901
True Skill Statistic   = 0.448
Prevalence Identified  = 0.809
Balanced Accuracy      = 0.724

95% Confidence Intervals:
  Precision     = [0.785, 0.950]
  Sensitivity   = [0.804, 0.961]
  Specificity   = [0.291, 0.768]
  TSS           = [0.096, 0.728]

## Temporal analysis on the testing dataset. 

In [ ]:
# =============================================================================
# TEMPORAL-RANGE VARIABILITY ANALYSIS
# =============================================================================
#
# Purpose:
#   1. Quantify how long each deer was monitored.
#   2. Determine whether monitoring duration varies across test animals.
#   3. Compare temporal coverage among CWD status and prediction outcomes.
#   4. Put the 75-anomalous-ping threshold into units of approximate days.
#
# Assumption:
#   Approximately 4 GPS fixes are expected per day.
#   Therefore, 75 anomalous fixes represent approximately 18.75 days.
# =============================================================================

# ── SETTINGS ──────────────────────────────────────────────────────────────────
EXPECTED_PINGS_PER_DAY = 4
ANOMALY_PING_THRESHOLD = 75

ANOMALY_DAYS_EQUIVALENT = (
    ANOMALY_PING_THRESHOLD / EXPECTED_PINGS_PER_DAY
)

temporal_id_out_csv = str(test_feat_path).replace(
    ".csv",
    "_temporal_range_by_id.csv"
)

temporal_group_out_csv = str(test_feat_path).replace(
    ".csv",
    "_temporal_range_group_summary.csv"
)

print("\n" + "=" * 78)
print("TEMPORAL-RANGE VARIABILITY ANALYSIS")
print("=" * 78)
print(
    f"Anomaly requirement: {ANOMALY_PING_THRESHOLD} pings "
    f"≈ {ANOMALY_DAYS_EQUIVALENT:.2f} days "
    f"at {EXPECTED_PINGS_PER_DAY} expected pings/day."
)

# ── CHECK DATETIME COLUMN ─────────────────────────────────────────────────────
if "datetime" not in df.columns:
    raise ValueError(
        "The temporal-range analysis requires a 'datetime' column."
    )

# Work from the rows used by the model
temporal_rows = rows.copy()

temporal_rows["datetime"] = pd.to_datetime(
    temporal_rows["datetime"],
    errors="coerce"
)

n_missing_datetime = temporal_rows["datetime"].isna().sum()

if n_missing_datetime > 0:
    print(
        f"⚠️ Warning: {n_missing_datetime:,} rows have missing or invalid "
        "datetime values and will not contribute to date-based calculations."
    )

valid_temporal_rows = temporal_rows.dropna(
    subset=["datetime"]
).copy()

if valid_temporal_rows.empty:
    raise ValueError(
        "No valid datetime values were available for temporal analysis."
    )

# Create a calendar-date variable so we can count unique monitored days
valid_temporal_rows["calendar_date"] = (
    valid_temporal_rows["datetime"].dt.normalize()
)

# ── ID-LEVEL TEMPORAL SUMMARY ─────────────────────────────────────────────────
temporal_by_id = (
    valid_temporal_rows
    .groupby("id", sort=False)
    .agg(
        first_datetime=("datetime", "min"),
        last_datetime=("datetime", "max"),
        n_rows_temporal=("datetime", "size"),
        n_observed_days=("calendar_date", "nunique"),
        n_anomalous_pings=("row_flag", "sum")
    )
    .reset_index()
)

# Exact elapsed time between first and last recorded fixes
temporal_by_id["temporal_range_hours"] = (
    temporal_by_id["last_datetime"]
    - temporal_by_id["first_datetime"]
).dt.total_seconds() / 3600

temporal_by_id["temporal_range_days_exact"] = (
    temporal_by_id["temporal_range_hours"] / 24
)

# Inclusive calendar duration:
# If observations occur on January 1 and January 2, this equals 2 days.
temporal_by_id["calendar_span_days"] = (
    temporal_by_id["last_datetime"].dt.normalize()
    - temporal_by_id["first_datetime"].dt.normalize()
).dt.days + 1

# Number of fixes per day that actually contained observations
temporal_by_id["observed_pings_per_day"] = (
    temporal_by_id["n_rows_temporal"]
    / temporal_by_id["n_observed_days"].replace(0, np.nan)
)

# Number of fixes relative to the full inclusive date span
# This captures gaps in the temporal record.
temporal_by_id["span_adjusted_pings_per_day"] = (
    temporal_by_id["n_rows_temporal"]
    / temporal_by_id["calendar_span_days"].replace(0, np.nan)
)

# Percentage of calendar days between the first and last record
# that contain at least one GPS observation.
temporal_by_id["day_coverage_pct"] = (
    100
    * temporal_by_id["n_observed_days"]
    / temporal_by_id["calendar_span_days"].replace(0, np.nan)
)

# Approximate monitoring duration represented by the number of rows,
# assuming four expected fixes per day.
temporal_by_id["fix_equivalent_days"] = (
    temporal_by_id["n_rows_temporal"]
    / EXPECTED_PINGS_PER_DAY
)

# Approximate duration represented by anomalous fixes
temporal_by_id["anomalous_fix_equivalent_days"] = (
    temporal_by_id["n_anomalous_pings"]
    / EXPECTED_PINGS_PER_DAY
)

# Minimum possible proportion of the ID's observations that must be anomalous
# to reach 75 anomalous fixes.
temporal_by_id["required_anomaly_fraction"] = (
    ANOMALY_PING_THRESHOLD
    / temporal_by_id["n_rows_temporal"].replace(0, np.nan)
)

# Whether the animal has at least 75 total rows.
# An animal with fewer than 75 total rows could never satisfy the threshold.
temporal_by_id["can_reach_75_from_total_rows"] = (
    temporal_by_id["n_rows_temporal"] >= ANOMALY_PING_THRESHOLD
).astype(int)

# Whether the observed monitoring period spans at least the approximate
# 18.75 days represented by 75 fixes at four fixes/day.
temporal_by_id["has_18_75_day_span"] = (
    temporal_by_id["calendar_span_days"]
    >= ANOMALY_DAYS_EQUIVALENT
).astype(int)

# Whether the ID actually met the count threshold
temporal_by_id["met_75_anomaly_count"] = (
    temporal_by_id["n_anomalous_pings"]
    >= ANOMALY_PING_THRESHOLD
).astype(int)

# ── MERGE STATUS, PREDICTION, AND ERROR INFORMATION ───────────────────────────
pos_set = set(actual_cwd_instances_test)
neg_set = set(unaffected_cwd_instances_test)
pred_set = set(pred_ids)

temporal_by_id["actual_status"] = temporal_by_id["id"].map(
    lambda x: (
        "CWD+"
        if x in pos_set
        else "CWD-"
        if x in neg_set
        else "Unknown"
    )
)

temporal_by_id["predicted_status"] = temporal_by_id["id"].map(
    lambda x: "Predicted CWD+" if x in pred_set else "Predicted CWD-"
)

def assign_prediction_group(animal_id):
    """Assign TP, FN, FP, TN, or Unknown based on actual and predicted status."""
    actual_positive = animal_id in pos_set
    actual_negative = animal_id in neg_set
    predicted_positive = animal_id in pred_set

    if actual_positive and predicted_positive:
        return "TP"
    elif actual_positive and not predicted_positive:
        return "FN"
    elif actual_negative and predicted_positive:
        return "FP"
    elif actual_negative and not predicted_positive:
        return "TN"
    else:
        return "Unknown"

temporal_by_id["prediction_group"] = temporal_by_id["id"].map(
    assign_prediction_group
)

# Merge relevant values already calculated in ids_df
temporal_by_id = temporal_by_id.merge(
    ids_df[
        [
            "id",
            "n_rows",
            "n_flags",
            "flag_frac",
            "rule_aligned_score",
            "id_max_error",
            "id_mean_error",
            "eligible",
            "is_positive_id"
        ]
    ],
    on="id",
    how="left",
    validate="one_to_one"
)

# Sanity check: temporal count should match model row count
temporal_by_id["row_count_difference"] = (
    temporal_by_id["n_rows_temporal"]
    - temporal_by_id["n_rows"]
)

if not (temporal_by_id["row_count_difference"] == 0).all():
    print(
        "⚠️ Some IDs have temporal row counts that differ from the model row "
        "counts, likely because of missing datetime values."
    )

# Arrange columns in a reviewer-friendly order
temporal_columns = [
    "id",
    "actual_status",
    "predicted_status",
    "prediction_group",
    "first_datetime",
    "last_datetime",
    "temporal_range_days_exact",
    "calendar_span_days",
    "n_observed_days",
    "day_coverage_pct",
    "n_rows_temporal",
    "observed_pings_per_day",
    "span_adjusted_pings_per_day",
    "fix_equivalent_days",
    "n_anomalous_pings",
    "anomalous_fix_equivalent_days",
    "required_anomaly_fraction",
    "can_reach_75_from_total_rows",
    "has_18_75_day_span",
    "met_75_anomaly_count",
    "flag_frac",
    "rule_aligned_score",
    "id_mean_error",
    "id_max_error",
    "eligible",
    "is_positive_id",
    "row_count_difference"
]

temporal_by_id = temporal_by_id[temporal_columns].sort_values(
    ["actual_status", "prediction_group", "id"]
).reset_index(drop=True)

# ── OVERALL TEMPORAL VARIABILITY ──────────────────────────────────────────────
def safe_iqr(series):
    """Return the interquartile range while ignoring missing values."""
    clean = pd.to_numeric(series, errors="coerce").dropna()

    if clean.empty:
        return np.nan

    return clean.quantile(0.75) - clean.quantile(0.25)


def safe_cv(series):
    """
    Coefficient of variation:
    standard deviation divided by the mean.
    """
    clean = pd.to_numeric(series, errors="coerce").dropna()

    if len(clean) < 2:
        return np.nan

    mean_value = clean.mean()

    if mean_value == 0:
        return np.nan

    return clean.std(ddof=1) / mean_value


def describe_temporal_group(group_df, group_name):
    """Create one summary row for a temporal-analysis group."""
    duration = group_df["calendar_span_days"]
    observed_days = group_df["n_observed_days"]
    fixes = group_df["n_rows_temporal"]
    fix_rate = group_df["observed_pings_per_day"]
    coverage = group_df["day_coverage_pct"]

    return {
        "group": group_name,
        "n_ids": len(group_df),

        "duration_mean_days": duration.mean(),
        "duration_sd_days": duration.std(ddof=1),
        "duration_median_days": duration.median(),
        "duration_min_days": duration.min(),
        "duration_q25_days": duration.quantile(0.25),
        "duration_q75_days": duration.quantile(0.75),
        "duration_max_days": duration.max(),
        "duration_iqr_days": safe_iqr(duration),
        "duration_cv": safe_cv(duration),

        "observed_days_mean": observed_days.mean(),
        "observed_days_median": observed_days.median(),

        "fixes_mean": fixes.mean(),
        "fixes_median": fixes.median(),
        "fixes_min": fixes.min(),
        "fixes_max": fixes.max(),

        "observed_pings_per_day_mean": fix_rate.mean(),
        "observed_pings_per_day_median": fix_rate.median(),

        "day_coverage_mean_pct": coverage.mean(),
        "day_coverage_median_pct": coverage.median(),

        "pct_can_reach_75_rows": (
            100 * group_df["can_reach_75_from_total_rows"].mean()
        ),
        "pct_with_18_75_day_span": (
            100 * group_df["has_18_75_day_span"].mean()
        ),
        "pct_meeting_75_anomaly_count": (
            100 * group_df["met_75_anomaly_count"].mean()
        )
    }


summary_rows = []

# Overall
summary_rows.append(
    describe_temporal_group(
        temporal_by_id,
        "All test IDs"
    )
)

# Actual disease status
for status in ["CWD+", "CWD-"]:
    subset = temporal_by_id[
        temporal_by_id["actual_status"] == status
    ]

    if not subset.empty:
        summary_rows.append(
            describe_temporal_group(
                subset,
                status
            )
        )

# Prediction outcome
for outcome in ["TP", "FN", "FP", "TN"]:
    subset = temporal_by_id[
        temporal_by_id["prediction_group"] == outcome
    ]

    if not subset.empty:
        summary_rows.append(
            describe_temporal_group(
                subset,
                outcome
            )
        )

temporal_group_summary = pd.DataFrame(summary_rows)

# Round numeric summary columns for easier viewing
numeric_summary_cols = temporal_group_summary.select_dtypes(
    include=[np.number]
).columns

temporal_group_summary[numeric_summary_cols] = (
    temporal_group_summary[numeric_summary_cols].round(3)
)

# ── PRINT MAIN RESULTS ────────────────────────────────────────────────────────
duration = temporal_by_id["calendar_span_days"]

print("\n=== Overall Temporal Coverage ===")
print(f"Number of test IDs: {len(temporal_by_id)}")
print(
    f"Mean temporal range:   {duration.mean():.1f} days "
    f"(SD = {duration.std(ddof=1):.1f})"
)
print(
    f"Median temporal range: {duration.median():.1f} days "
    f"(IQR = {duration.quantile(0.25):.1f}–"
    f"{duration.quantile(0.75):.1f})"
)
print(
    f"Minimum–maximum:       {duration.min():.1f}–"
    f"{duration.max():.1f} days"
)
print(
    f"Coefficient of variation: {safe_cv(duration):.3f}"
)

print("\n=== Fix-Rate and Coverage Summary ===")
print(
    "Median observed fixes per monitored day: "
    f"{temporal_by_id['observed_pings_per_day'].median():.2f}"
)
print(
    "Median date-span-adjusted fixes per day: "
    f"{temporal_by_id['span_adjusted_pings_per_day'].median():.2f}"
)
print(
    "Median percentage of calendar days represented: "
    f"{temporal_by_id['day_coverage_pct'].median():.1f}%"
)

n_cannot_reach = (
    temporal_by_id["can_reach_75_from_total_rows"] == 0
).sum()

n_short_span = (
    temporal_by_id["has_18_75_day_span"] == 0
).sum()

print("\n=== Anomaly-Threshold Temporal Context ===")
print(
    f"The {ANOMALY_PING_THRESHOLD}-ping threshold represents "
    f"approximately {ANOMALY_DAYS_EQUIVALENT:.2f} days at "
    f"{EXPECTED_PINGS_PER_DAY} fixes/day."
)
print(
    f"IDs with fewer than {ANOMALY_PING_THRESHOLD} total rows: "
    f"{n_cannot_reach} of {len(temporal_by_id)}"
)
print(
    f"IDs with monitoring spans shorter than "
    f"{ANOMALY_DAYS_EQUIVALENT:.2f} days: "
    f"{n_short_span} of {len(temporal_by_id)}"
)

# Print the most important comparison table
print("\n=== Temporal Range by Group ===")
display_columns = [
    "group",
    "n_ids",
    "duration_mean_days",
    "duration_sd_days",
    "duration_median_days",
    "duration_q25_days",
    "duration_q75_days",
    "duration_min_days",
    "duration_max_days",
    "duration_cv"
]

print(
    temporal_group_summary[display_columns]
    .to_string(index=False)
)

# ── TEMPORAL RANGE FOR CORRECT VS INCORRECT CLASSIFICATIONS ───────────────────
temporal_by_id["classification_result"] = np.where(
    temporal_by_id["prediction_group"].isin(["TP", "TN"]),
    "Correct",
    np.where(
        temporal_by_id["prediction_group"].isin(["FN", "FP"]),
        "Incorrect",
        "Unknown"
    )
)

known_results = temporal_by_id[
    temporal_by_id["classification_result"].isin(
        ["Correct", "Incorrect"]
    )
].copy()

if not known_results.empty:
    correct_incorrect_summary = (
        known_results
        .groupby("classification_result")
        .agg(
            n_ids=("id", "nunique"),
            mean_duration_days=("calendar_span_days", "mean"),
            median_duration_days=("calendar_span_days", "median"),
            min_duration_days=("calendar_span_days", "min"),
            max_duration_days=("calendar_span_days", "max"),
            mean_fix_count=("n_rows_temporal", "mean"),
            median_fix_count=("n_rows_temporal", "median"),
            mean_observed_fix_rate=(
                "observed_pings_per_day",
                "mean"
            ),
            mean_day_coverage_pct=("day_coverage_pct", "mean")
        )
        .reset_index()
    )

    print("\n=== Correct vs. Incorrect Classification Coverage ===")
    print(
        correct_incorrect_summary.round(2).to_string(index=False)
    )

# ── CWD+ ONLY: COMPARE TRUE POSITIVES WITH FALSE NEGATIVES ────────────────────
cwd_positive_temporal = temporal_by_id[
    temporal_by_id["actual_status"] == "CWD+"
].copy()

tp_fn_temporal_summary = (
    cwd_positive_temporal[
        cwd_positive_temporal["prediction_group"].isin(["TP", "FN"])
    ]
    .groupby("prediction_group")
    .agg(
        n_ids=("id", "nunique"),
        mean_temporal_range_days=("calendar_span_days", "mean"),
        median_temporal_range_days=("calendar_span_days", "median"),
        min_temporal_range_days=("calendar_span_days", "min"),
        max_temporal_range_days=("calendar_span_days", "max"),
        mean_total_pings=("n_rows_temporal", "mean"),
        median_total_pings=("n_rows_temporal", "median"),
        mean_anomalous_pings=("n_anomalous_pings", "mean"),
        median_anomalous_pings=("n_anomalous_pings", "median"),
        mean_flag_fraction=("flag_frac", "mean"),
        mean_day_coverage_pct=("day_coverage_pct", "mean")
    )
    .reset_index()
)

if not tp_fn_temporal_summary.empty:
    print("\n=== CWD+ Temporal Coverage: TP vs. FN ===")
    print(
        tp_fn_temporal_summary.round(3).to_string(index=False)
    )

# ── SIMPLE CORRELATIONS ───────────────────────────────────────────────────────
# These do not establish causation. They only show whether longer temporal
# records tend to contain more total or anomalous observations.
correlation_columns = [
    "calendar_span_days",
    "n_observed_days",
    "n_rows_temporal",
    "n_anomalous_pings",
    "flag_frac",
    "rule_aligned_score"
]

temporal_correlations = (
    temporal_by_id[correlation_columns]
    .corr(method="spearman")
    .round(3)
)

print("\n=== Spearman Temporal Correlations ===")
print(temporal_correlations.to_string())

# ── IDENTIFY SHORTEST AND LONGEST RECORDS ─────────────────────────────────────
print("\n=== Five Shortest Temporal Records ===")
print(
    temporal_by_id[
        [
            "id",
            "actual_status",
            "prediction_group",
            "calendar_span_days",
            "n_observed_days",
            "n_rows_temporal",
            "n_anomalous_pings",
            "flag_frac"
        ]
    ]
    .sort_values("calendar_span_days")
    .head(5)
    .round(3)
    .to_string(index=False)
)

print("\n=== Five Longest Temporal Records ===")
print(
    temporal_by_id[
        [
            "id",
            "actual_status",
            "prediction_group",
            "calendar_span_days",
            "n_observed_days",
            "n_rows_temporal",
            "n_anomalous_pings",
            "flag_frac"
        ]
    ]
    .sort_values("calendar_span_days", ascending=False)
    .head(5)
    .round(3)
    .to_string(index=False)
)

# ── SAVE OUTPUTS ──────────────────────────────────────────────────────────────
temporal_by_id.to_csv(
    temporal_id_out_csv,
    index=False
)

temporal_group_summary.to_csv(
    temporal_group_out_csv,
    index=False
)

print("\n✅ Temporal outputs written:")
print(f"   {temporal_id_out_csv}")
print(f"   {temporal_group_out_csv}")

In [ ]:
# Define the actual CWD instances and unaffected CWD instances
actual_cwd_instances = [1, 2, 6, 8, 11, 12, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 30, 31, 32, 35, 36, 37, 40, 47, 48, 53, 57, 58, 61, 62, 64, 65, 67, 68, 69, 71, 72, 74, 79, 83, 90, 91, 92, 93, 94, 96, 102, 105, 113, 114, 121, 122, 125, 130, 141]
unaffected_cwd_instances = [5, 9, 14, 22, 26, 29, 44, 55, 66, 95, 108, 118, 129]

# You already have high_anomaly_ids from Step 1
print("High anomaly IDs:", pred_ids)

# Convert high_anomaly_ids to a set for easier comparison
high_deviation_ids = set(pred_ids)

# Calculate True Positives (TP), False Negatives (FN), False Positives (FP), and True Negatives (TN)
TP = len([ID for ID in actual_cwd_instances if ID in pred_ids])
FN = len([ID for ID in actual_cwd_instances if ID not in pred_ids])
FP = len([ID for ID in unaffected_cwd_instances if ID in pred_ids])
TN = len([ID for ID in unaffected_cwd_instances if ID not in pred_ids])

# Calculate False Positive Rate (FPR), False Negative Rate (FNR), and Accuracy
FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
FNR = FN / (FN + TP) if (FN + TP) > 0 else 0
accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0

# Print the results
print(f"True Positives (TP): {TP}")
print(f"False Negatives (FN): {FN}")
print(f"False Positives (FP): {FP}")
print(f"True Negatives (TN): {TN}")
print(f"False Positive Rate (FPR): {FPR:.2f}")
print(f"False Negative Rate (FNR): {FNR:.2f}")
print(f"Accuracy: {accuracy:.2f}")



In [ ]:
# === Stage label definitions for 60/40 ===
ACTUAL_POS_IDS = {1,2,6,8,11,12,15,16,17,18,19,20,21,23,24,25,
                  30,31,32,35,36,37,40,47,48,53,57,58,61,62,64,
                  65,67,68,69,70,71,72,73,74,79,80,83,90,91,92,
                  93,94,96,102,105,107,113,114,116,121,122,125,130,141}
BECAME_POS_IDS = {6,11,19,20,25,32,40,48,62,74,105,122,125,47,53,
                  73,80,92,107,116,135}   # converters in your dataset

test_ids  = set(df["id"].unique())
early_ids = sorted(BECAME_POS_IDS & test_ids)
late_ids  = sorted((ACTUAL_POS_IDS & test_ids) - set(early_ids))
neg_ids   = sorted(test_ids - ACTUAL_POS_IDS)   # the 40% negatives in test

# === Compare AE predictions vs stage groups ===
pred_ids_set = set(pred_ids)

early_detected = pred_ids_set & set(early_ids)
late_detected  = pred_ids_set & set(late_ids)
neg_detected   = pred_ids_set & set(neg_ids)

print("\n=== AE detection by stage group (60/40 test) ===")
print(f"Early (converters): detected {len(early_detected)} / {len(early_ids)} → {sorted(early_detected)}")
print(f"Late (already positive): detected {len(late_detected)} / {len(late_ids)} → {sorted(late_detected)}")
print(f"Negatives: flagged {len(neg_detected)} / {len(neg_ids)} → {sorted(neg_detected)}")

# (Optional) add stage_label column to ids_df
def assign_stage_label(i):
    if i in early_ids:
        return "early"
    elif i in late_ids:
        return "late"
    elif i in neg_ids:
        return "negative"
    else:
        return "unknown"

ids_df["stage_label"] = ids_df["id"].apply(assign_stage_label)

# Quick group-wise detection summary
summary = ids_df.groupby("stage_label")["is_positive_id"].agg(
    detected="sum",
    total="count",
    detection_rate=lambda x: 100 * x.mean()
)
print("\n=== Detection summary by stage group ===")
print(summary)


In [ ]:
id_summary = df.groupby("id").agg({
    "datetime": ["min", "max", "count"],
    "id": "count"
}).droplevel(0, axis=1).rename(columns={"min": "start", "max": "end", "count": "n_rows"})
id_summary["duration_hours"] = (id_summary["end"] - id_summary["start"]).dt.total_seconds() / 3600
id_summary.sort_values("duration_hours", ascending=False).head(10)

id_summary.head()


In [ ]:
# ===== Binned (by ID duration in days) metrics & counts — SAME 2×3 COMPOSITION =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 0) Load full test and get per-ID duration (days)
df_full = pd.read_csv(test_feat_path)
time_col = "date_time" if "date_time" in df_full.columns else ("datetime" if "datetime" in df_full.columns else None)
if time_col is None:
    raise KeyError("No 'date_time' or 'datetime' column in test CSV.")
df_full[time_col] = pd.to_datetime(df_full[time_col], errors="coerce")

id_start = df_full.groupby("id")[time_col].min()
id_end   = df_full.groupby("id")[time_col].max()
id_duration_days = (id_end - id_start).dt.total_seconds() / 86400.0

# 1) Quantile bins on DAYS (equal-count), robust to ties/small N
N_BINS = 7
q = min(N_BINS, max(1, id_duration_days.dropna().nunique()))
cats, edges = pd.qcut(id_duration_days, q=q, retbins=True, duplicates='drop')

def _label(left, right):
    L = int(np.floor(left)); R = int(np.ceil(right))
    return f"{L}–{R} days" if R > L else f"{L} days"

intervals  = cats.cat.categories
bin_labels = [_label(iv.left, iv.right) for iv in intervals]
cats = cats.cat.rename_categories(bin_labels)

# Map bin label to each ID and attach to ids_df
id_to_bin = pd.Series(cats.values, index=id_duration_days.index).to_dict()
ids_df = ids_df.copy()
ids_df["bin"] = ids_df["id"].map(id_to_bin)

# 2) Helpers
def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = z * np.sqrt((p*(1-p)/n) + (z*z/(4*n*n))) / denom
    return (center - half, center + half)

def counts_to_metrics(TN, FP, FN, TP):
    N = TP + TN + FP + FN
    acc  = (TP + TN) / N if N else np.nan
    prec = TP / (TP + FP) if (TP + FP) else np.nan
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    spec = TN / (TN + FP) if (TN + FP) else np.nan
    tss  = sens + spec - 1 if np.isfinite(sens) and np.isfinite(spec) else np.nan
    acc_lo, acc_hi   = wilson_ci(TP + TN, N) if N else (np.nan, np.nan)
    prec_lo, prec_hi = wilson_ci(TP, TP + FP) if (TP + FP) else (np.nan, np.nan)
    sens_lo, sens_hi = wilson_ci(TP, TP + FN) if (TP + FN) else (np.nan, np.nan)
    spec_lo, spec_hi = wilson_ci(TN, TN + FP) if (TN + FP) else (np.nan, np.nan)
    tss_lo, tss_hi   = (sens_lo + spec_lo - 1, sens_hi + spec_hi - 1) if np.isfinite(sens_lo) and np.isfinite(spec_lo) else (np.nan, np.nan)
    return dict(accuracy=acc, accuracy_lo=acc_lo, accuracy_hi=acc_hi,
                precision=prec, precision_lo=prec_lo, precision_hi=prec_hi,
                tss=tss, tss_lo=tss_lo, tss_hi=tss_hi)

# 3) Build per-bin summary using your *non-CV* predictions (pred_ids) + label lists
#    (If you’re in CV/ensemble mode, swap to ids_df['true_label_id'] and ids_df['id_pred'].)
summary = []
bin_order = bin_labels  # preserve quantile order

for b in bin_order:
    sub = ids_df[(ids_df["bin"] == b) & (ids_df["eligible"] == 1)]
    eval_ids = sub["id"].tolist()
    if len(eval_ids) == 0:
        summary.append({"bin": b, "n_eval": 0, "TP":0, "FP":0, "FN":0, "TN":0,
                        "accuracy": np.nan, "accuracy_lo": np.nan, "accuracy_hi": np.nan,
                        "precision": np.nan, "precision_lo": np.nan, "precision_hi": np.nan,
                        "tss": np.nan, "tss_lo": np.nan, "tss_hi": np.nan})
        continue

    # Ground truth & predictions from your lists
    pos_set = set(actual_cwd_instances_test)
    y_true = np.array([1 if i in pos_set else 0 for i in eval_ids])
    y_pred = np.array([1 if i in pred_ids else 0 for i in eval_ids])

    TN, FP, FN, TP = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    met = counts_to_metrics(TN, FP, FN, TP)

    summary.append({"bin": b, "n_eval": len(eval_ids), "TP":TP, "FP":FP, "FN":FN, "TN":TN, **met})

bin_df = pd.DataFrame(summary)
bin_df["bin"] = pd.Categorical(bin_df["bin"], categories=bin_order, ordered=True)
bin_df = bin_df.sort_values("bin")

print("\nPer-bin counts (eligible IDs):")
print(bin_df[["bin","n_eval","TP","FP","FN","TN"]].to_string(index=False))

# 4) Plot — SAME 2×3 COMPOSITION
def plot_bars_three(ddd: pd.DataFrame, title: str, fname: str):
    if ddd.empty:
        print("⚠️ Nothing to plot.")
        return
    ddd = ddd.copy()
    labels = ddd["bin"].astype(str).tolist()
    x = np.arange(len(labels), dtype=float)

    # Ensure columns exist
    for col in ["accuracy","accuracy_lo","accuracy_hi",
                "precision","precision_lo","precision_hi",
                "tss","tss_lo","tss_hi","TP","FP","FN","TN"]:
        if col not in ddd.columns: ddd[col] = np.nan

    fig, ax = plt.subplots(2, 3, figsize=(18, 10), sharex=False)

    panels = [
        ("accuracy","accuracy_lo","accuracy_hi","Accuracy", 0.0, 1.0, "#cfe1f2"),
        ("precision","precision_lo","precision_hi","Precision", 0.0, 1.0, "#cfe1f2"),
        ("tss","tss_lo","tss_hi","TSS", -1.0, 1.0, "#dcd3f2"),
    ]
    for i, (m, lo, hi, ttl, ymin, ymax, color) in enumerate(panels):
        center = ddd[m].to_numpy(float)
        lo_arr = center - ddd[lo].to_numpy(float)
        hi_arr = ddd[hi].to_numpy(float) - center
        msk = np.isfinite(center) & np.isfinite(lo_arr) & np.isfinite(hi_arr)

        # placeholders so bins always show
        ax[0,i].bar(x, np.nan_to_num(center, nan=0.0), color="#eeeeee", edgecolor="black",
                    linewidth=0.6, width=0.8, zorder=1)
        # real bars + error
        ax[0,i].bar(x[msk], center[msk], color=color, edgecolor="black",
                    linewidth=0.6, width=0.8, zorder=2)
        if np.any(msk):
            ax[0,i].errorbar(x[msk], center[msk], yerr=np.vstack([lo_arr[msk], hi_arr[msk]]),
                             fmt="none", ecolor="black", capsize=5, elinewidth=1.2, zorder=3)

        ax[0,i].set_title(ttl); ax[0,i].set_ylim(ymin, ymax)
        ax[0,i].set_xlim(-0.5, len(x)-0.5)
        ax[0,i].grid(axis="y", linestyle="--", alpha=0.35)
        ax[0,i].set_ylabel("Score")

    # Bottom row: TP/FP/FN; TN on separate axis
    for j, cm_m, col in [(0,"TP","#f6c0c0"), (1,"FP","#f6c0c0"), (2,"FN","#f6c0c0")]:
        vals = ddd[cm_m].fillna(0).to_numpy(int)
        ax[1,j].bar(x, vals, color=col, edgecolor="black", linewidth=0.6)
        ax[1,j].set_title(cm_m); ax[1,j].set_ylabel("Count")
        ax[1,j].set_xlim(-0.5, len(x)-0.5)
        ax[1,j].grid(axis="y", linestyle="--", alpha=0.35)

    fig.delaxes(ax[1,2])
    tn_ax = fig.add_subplot(2, 3, 6)
    tn_ax.bar(x, ddd["TN"].fillna(0).to_numpy(int), color="#cfeccf", edgecolor="black", linewidth=0.6)
    tn_ax.set_title("TN"); tn_ax.set_ylabel("Count")
    tn_ax.set_xlim(-0.5, len(x)-0.5); tn_ax.grid(axis="y", linestyle="--", alpha=0.35)

    for a in [ax[1,0], ax[1,1], tn_ax]:
        a.set_xticks(x); a.set_xticklabels(labels, rotation=30, ha="right")
    for a in ax[0,:]:
        a.set_xticks(x); a.set_xticklabels([])

    plt.suptitle("60/40 Experiment: Metrics & Confusion Counts by Duration Bin", fontsize=16, y=0.98)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    out_png = base / "ae_metrics_bins_by_duration_same_composition.png"
    out_pdf = base / "ae_metrics_bins_by_duration_same_composition.pdf"
    plt.savefig(out_png, bbox_inches="tight"); plt.savefig(out_pdf, bbox_inches="tight")
    print(f"📊 Saved: {out_png}\n📊 Saved: {out_pdf}")
    plt.show(); plt.close(fig)

plot_bars_three(bin_df, "60/40 Experiment: Metrics & Counts by Duration Bin", "bins_by_duration_same_composition")


In [ ]:
# ===== AE Feature Contributions — holdout-friendly (manuscript-ready, with TP/FN) =====
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
import joblib

# ----------------------------
# Config (point to your 100% holdout experiment)
# ----------------------------
base   = Path("<LOCAL_PATH>")
outdir = base / "ae_holdout_results"
outdir.mkdir(parents=True, exist_ok=True)

# Processed files & artifacts produced by your preprocessing pipeline
PROC_TEST = base / "test_processed_60_40.csv"
SCA_PATH  = base / "scaler_60_40.pkl"
FEAT_PATH = base / "feature_columns_60_40.pkl"

# If you want to predict reconstruction on-the-fly:
TRY_PREDICT_RECON = True   # set False if you define 'recon' upstream
BATCH = 128                # batch size for predict()

# Row-level flagging (if no 'row_flag' column exists in test)
ROW_FLAG_TAU  = None       # absolute threshold on total residual (sum of sq err) e.g., 3.5
ROW_FLAG_PCTL = 95.0       # percentile threshold fallback (top X% rows flagged)

# ID aggregation → predicted positives (used to form TP/FN etc.)
MIN_ROWS_PER_ID = 10
AGGREGATION_RULE = "OR"    # {"OR","AND","COUNT_ONLY","FRAC_ONLY"}
COUNT = 5
FRAC  = 0.05

# Ground-truth on holdout:
#   - If your test is all-positive and lacks labels ⇒ set ASSUME_TEST_ALL_POS=True
#   - If test has per-row or per-id label you trust (e.g., 'CWD_label') ⇒ set to False
ASSUME_TEST_ALL_POS = True

# ----------------------------
# Load processed test & feature list
# ----------------------------
df_test    = pd.read_csv(PROC_TEST)
feat_names = joblib.load(FEAT_PATH)  # exact order used for training
if not all(c in df_test.columns for c in feat_names):
    missing = [c for c in feat_names if c not in df_test.columns]
    raise KeyError(f"Processed test missing features: {missing[:10]}{' ...' if len(missing)>10 else ''}")

X_test   = df_test[feat_names].to_numpy(dtype=np.float32)
ids_rows = df_test["id"].to_numpy()
has_row_flag = "row_flag" in df_test.columns  # optional, from upstream pipeline

# ----------------------------
# Reconstructions & residuals
# ----------------------------
try:
    _ = recon  # use precomputed recon if provided
except NameError:
    if not TRY_PREDICT_RECON:
        raise ValueError("No 'recon' provided and TRY_PREDICT_RECON is False.")
    try:
        _ = autoencoder
    except NameError:
        raise ValueError("To predict 'recon' on the fly, define 'autoencoder' in the environment.")
    recon = autoencoder.predict(X_test, batch_size=BATCH, verbose=0)

recon = np.asarray(recon, float)
if X_test.shape != recon.shape:
    raise ValueError(f"Shape mismatch: X_test {X_test.shape} vs recon {recon.shape}")

residual_sq = (X_test - recon) ** 2     # (N_rows, n_features)
resid_total = residual_sq.sum(axis=1)   # per-row total residual (τ)

# ----------------------------
# Build the 'rows' frame for masks
# ----------------------------
rows = pd.DataFrame({"id": ids_rows})
if has_row_flag:
    rows["row_flag"] = df_test["row_flag"].astype(bool).to_numpy()
else:
    if ROW_FLAG_TAU is not None:
        rows["row_flag"] = resid_total > ROW_FLAG_TAU
    else:
        thr = np.nanpercentile(resid_total, ROW_FLAG_PCTL)
        rows["row_flag"] = resid_total >= thr

mask_all  = np.ones(len(rows), dtype=bool)
mask_flag = rows["row_flag"].to_numpy(bool)
mask_eval = mask_all.copy()  # whole holdout

# ----------------------------
# Per-feature contribution helper (→ %)
# ----------------------------
def contrib_pct(rsq, mask):
    if rsq.size == 0 or not np.any(mask):
        return np.full(rsq.shape[1], np.nan, dtype=float)
    m = np.nanmean(rsq[mask], axis=0)
    s = np.nansum(m)
    return (m / s) * 100.0 if s > 0 else np.full(rsq.shape[1], np.nan, dtype=float)

contrib_all   = contrib_pct(residual_sq, mask_all)
contrib_flag  = contrib_pct(residual_sq, mask_flag)
contrib_eval  = contrib_pct(residual_sq, mask_eval)

# ----------------------------
# Erraticness across IDs (median per-feature SD in %-points)
# ----------------------------
id_to_rows = defaultdict(list)
for r, i in enumerate(rows["id"].to_numpy()):
    id_to_rows[i].append(r)

def per_id_contrib_stats(mask):
    per_id = []
    for i, idxs in id_to_rows.items():
        idxs = np.asarray(idxs, dtype=int)
        idxs = idxs[mask[idxs]]
        if idxs.size == 0:
            continue
        m = np.nanmean(residual_sq[idxs], axis=0)
        s = np.nansum(m)
        per_id.append((i, (m / s * 100.0) if s > 0 else np.full(residual_sq.shape[1], np.nan)))
    if not per_id:
        return dict(median_feature_sd_pp=np.nan, n_ids=0)
    M = np.vstack([v for _, v in per_id])  # (n_ids, n_features)
    std_per_feat_pp = np.nanstd(M, axis=0)
    return dict(median_feature_sd_pp=float(np.nanmedian(std_per_feat_pp)), n_ids=len(per_id))

err_all  = per_id_contrib_stats(mask_all)
err_flag = per_id_contrib_stats(mask_flag)
err_eval = per_id_contrib_stats(mask_eval)

# ----------------------------
# Simple concentration proxies
# ----------------------------
def topk_share_pct(arr, k=3):
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    if a.size == 0: return np.nan
    a_sorted = np.sort(a)[::-1]
    return float(np.sum(a_sorted[:min(k, a_sorted.size)]))

top3_all,  top5_all  = topk_share_pct(contrib_all, 3), topk_share_pct(contrib_all, 5)
top3_flag, top5_flag = topk_share_pct(contrib_flag, 3), topk_share_pct(contrib_flag, 5)
top3_eval, top5_eval = topk_share_pct(contrib_eval, 3), topk_share_pct(contrib_eval, 5)

# ----------------------------
# Per-feature table
# ----------------------------
df_contrib = pd.DataFrame({
    "feature":    feat_names,
    "All_%":      contrib_all,
    "Flagged_%":  contrib_flag,
    "Eval_%":     contrib_eval,
})
df_contrib["Delta_Flagged_minus_All_pp"] = df_contrib["Flagged_%"] - df_contrib["All_%"]

print("\n=== AE Feature Contributions (% of reconstruction error) — Holdout ===")
print(df_contrib.sort_values("Flagged_%", ascending=False).head(15).to_string(index=False,
      float_format=lambda x: f"{x:.2f}"))

summary_tbl = pd.DataFrame([
    {"Group":"All rows",    "Top3_share_%": top3_all,  "Top5_share_%": top5_all,  "Median_feat_SD_pp": err_all["median_feature_sd_pp"],  "n_ids": err_all["n_ids"]},
    {"Group":"Flagged rows","Top3_share_%": top3_flag, "Top5_share_%": top5_flag, "Median_feat_SD_pp": err_flag["median_feature_sd_pp"], "n_ids": err_flag["n_ids"]},
    {"Group":"Eval IDs",    "Top3_share_%": top3_eval, "Top5_share_%": top5_eval, "Median_feat_SD_pp": err_eval["median_feature_sd_pp"], "n_ids": err_eval["n_ids"]},
])
summary_tbl["ΔFlagged−All_Top3_pp"] = summary_tbl.loc[1,"Top3_share_%"] - summary_tbl.loc[0,"Top3_share_%"]
summary_tbl["ΔFlagged−All_Top5_pp"] = summary_tbl.loc[1,"Top5_share_%"] - summary_tbl.loc[0,"Top5_share_%"]
summary_tbl["ΔFlagged−All_SD_pp"]   = summary_tbl.loc[1,"Median_feat_SD_pp"] - summary_tbl.loc[0,"Median_feat_SD_pp"]

print("\n=== Simple concentration & variability summary (holdout) ===")
print(summary_tbl.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# ----------------------------
# Category roll-up (movement story)
# ----------------------------
def categorize(fname: str) -> str:
    f = fname.lower()
    if "velo" in f: return "velocity"
    if "angle" in f: return "turning-angle"
    if "sinu" in f:  return "sinuosity"
    if "skew" in f or "kurt" in f: return "distributional-moment"
    if "rolling" in f or "mean" in f or "sd" in f: return "rolling-statistic"
    if f in {"low","est","high"} or "model" in f: return "model-derived"
    if "season" in f: return "season"
    return "other"

cat = pd.DataFrame({
    "feature":  feat_names,
    "category": [categorize(f) for f in feat_names],
    "All_%":    contrib_all,
    "Flagged_%":contrib_flag,
    "Eval_%":   contrib_eval
})
cat_roll = (cat.groupby("category")[["All_%","Flagged_%","Eval_%"]]
              .sum().sort_values("Flagged_%", ascending=False))
cat_roll["Delta_Flagged_minus_All_pp"] = cat_roll["Flagged_%"] - cat_roll["All_%"]

print("\n=== Category roll-up (% of recon error) — Holdout ===")
print(cat_roll.to_string(float_format=lambda x: f"{x:.1f}"))

# ----------------------------
# Build TP / FP / FN / TN partitions (ID-level)
# ----------------------------
# 1) Ground-truth ID sets
if ("pos_set" in globals()) and ("neg_set" in globals()):
    # Use externally provided sets if you already defined them
    pos_set, neg_set = set(pos_set), set(neg_set)
else:
    if (not ASSUME_TEST_ALL_POS) and ("CWD_label" in df_test.columns):
        # Use ID-level labels from df_test if present
        id_lab = df_test.groupby("id")["CWD_label"].max()
        pos_set = set(id_lab.index[id_lab == 1])
        neg_set = set(id_lab.index[id_lab == 0])
    else:
        # Fallback: holdout contains only positives
        pos_set = set(df_test["id"].unique())
        neg_set = set()

# 2) Aggregate row_flags → predicted positive IDs
def aggregate_ids_from_rowflags(row_flags_bool, ids_series,
                                MIN_ROWS_PER_ID=MIN_ROWS_PER_ID,
                                AGGREGATION_RULE=AGGREGATION_RULE,
                                COUNT=COUNT, FRAC=FRAC):
    tmp = pd.DataFrame({"id": ids_series.values, "row_flag": row_flags_bool.astype(int)})
    g = tmp.groupby("id", sort=False)
    n_rows = g.size()
    n_flags = g["row_flag"].sum()
    flag_fr = n_flags / n_rows

    eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
    n_flags_elig = n_flags.reindex(eligible_idx)
    flag_fr_elig = flag_fr.reindex(eligible_idx)

    if AGGREGATION_RULE == "AND":
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index) & set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    elif AGGREGATION_RULE == "FRAC_ONLY":
        pred = set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    elif AGGREGATION_RULE == "COUNT_ONLY":
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index)
    else:  # "OR"
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index) | set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    return sorted(pred), eligible_idx

pred_ids = globals().get("pred_ids", None)
if pred_ids is None:
    pred_ids, eligible_idx = aggregate_ids_from_rowflags(rows["row_flag"].to_numpy(bool), rows["id"])
else:
    pred_ids = sorted(set(pred_ids))
    eligible_idx = rows["id"].unique()

# 3) Confusion partitions on eligible IDs
eval_ids = sorted((pos_set | neg_set) & set(eligible_idx))
TP_ids = [i for i in eval_ids if (i in pos_set) and (i in pred_ids)]
FP_ids = [i for i in eval_ids if (i in neg_set) and (i in pred_ids)]
FN_ids = [i for i in eval_ids if (i in pos_set) and (i not in pred_ids)]
TN_ids = [i for i in eval_ids if (i in neg_set) and (i not in pred_ids)]

rid = rows["id"]
m_tp = rid.isin(TP_ids).to_numpy()
m_fp = rid.isin(FP_ids).to_numpy()
m_fn = rid.isin(FN_ids).to_numpy()
m_tn = rid.isin(TN_ids).to_numpy()

# ----------------------------
# Per-class contribution tables
# ----------------------------
def table_from_mask(mask, label):
    pct = contrib_pct(residual_sq, mask)
    return (pd.DataFrame({"feature": feat_names, f"{label}_%": pct})
              .sort_values(f"{label}_%", ascending=False).reset_index(drop=True))

tp_tbl = table_from_mask(m_tp, "TP")
fp_tbl = table_from_mask(m_fp, "FP")
fn_tbl = table_from_mask(m_fn, "FN")
tn_tbl = table_from_mask(m_tn, "TN")

# Print top-10s with Δ vs All (guard empty classes)
all_series = pd.Series(contrib_all, index=feat_names, name="All_%")
def top_with_delta(df, label, k=10):
    if df[f"{label}_%"].notna().any():
        out = (df.head(k)
                 .merge(all_series.reset_index().rename(columns={"index":"feature"}),
                        on="feature", how="left"))
        out[f"Delta_{label}_minus_All_pp"] = out[f"{label}_%"] - out["All_%"]
        print(f"\n=== Top {k} features — {label} ===")
        print(out.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    else:
        print(f"\n=== Top {k} features — {label} ===\n(no rows)")

top_with_delta(tp_tbl,"TP"); top_with_delta(fp_tbl,"FP")
top_with_delta(fn_tbl,"FN"); top_with_delta(tn_tbl,"TN")

def table_from_mask(mask, label):
    pct = contrib_pct(residual_sq, mask)
    return (pd.DataFrame({"feature": feat_names, f"{label}_%": pct})
              .sort_values(f"{label}_%", ascending=False).reset_index(drop=True))

def top_with_delta(df, label, k=10):
    if df[f"{label}_%"].notna().any():
        out = (df.head(k)
                 .merge(all_series.reset_index().rename(columns={"index":"feature"}),
                        on="feature", how="left"))
        out[f"Delta_{label}_minus_All_pp"] = out[f"{label}_%"] - out["All_%"]
        print(f"\n=== Top {k} features — {label} ===")
        print(out.to_string(index=False, float_format=lambda x: f"{x:.2f}"))


# Save per-class CSVs
tp_tbl.to_csv(outdir / "ae_holdout_feat_contrib_tp.csv", index=False)
fp_tbl.to_csv(outdir / "ae_holdout_feat_contrib_fp.csv", index=False)
fn_tbl.to_csv(outdir / "ae_holdout_feat_contrib_fn.csv", index=False)
tn_tbl.to_csv(outdir / "ae_holdout_feat_contrib_tn.csv", index=False)

# 🔄 Save ALL per-class features (not top-10)
for label, df in {"TP": tp_tbl, "FP": fp_tbl, "FN": fn_tbl, "TN": tn_tbl}.items():
    if df is not None and not df.empty:
        # Merge with All_% and compute delta for every feature
        merged = df.merge(all_series.reset_index().rename(columns={"index": "feature"}),
                          on="feature", how="left")
        merged[f"Delta_{label}_minus_All_pp"] = merged[f"{label}_%"] - merged["All_%"]
        merged.to_csv(outdir / f"ae_all_features_{label}.csv", index=False)
        print(f"✅ Saved full feature list for {label}: {outdir}/ae_all_features_{label}.csv")


# ----------------------------
# Movement–residual linkage (inverse-scaling numeric features only)
# ----------------------------
try:
    scaler = joblib.load(SCA_PATH)
    if not hasattr(scaler, "inverse_transform"):
        raise TypeError("Loaded scaler has no inverse_transform")
    num_cols = [c for c in feat_names if c in getattr(scaler, "feature_names_in_", [])]
    cat_cols = [c for c in feat_names if c not in num_cols]

    X_num = df_test[num_cols].to_numpy()
    X_cat = df_test[cat_cols].to_numpy() if cat_cols else np.empty((len(df_test), 0))
    X_num_raw = scaler.inverse_transform(X_num)

    df_raw = pd.DataFrame(np.hstack([X_num_raw, X_cat]), columns=num_cols + cat_cols)
    units_label = "raw units"
except Exception as e:
    print(f"[warn] Using scaled units — {e}")
    df_raw = df_test[feat_names].copy()
    units_label = "scaled units"

# Add IDs + row flags
df_raw["id"] = rows["id"].values
df_raw["row_flag"] = rows["row_flag"].values

# Collapsed movement metrics
df_raw["speed"]   = df_raw.filter(like="velo").mean(axis=1)
df_raw["turning"] = df_raw.filter(like="angle").mean(axis=1)
df_raw["sinu"]    = df_raw.filter(like="sinu").mean(axis=1)

def group_summary(mask, label):
    tmp = df_raw.loc[mask]
    return pd.Series({
        "n_rows":       int(len(tmp)),
        "mean_speed":   float(tmp["speed"].mean())   if len(tmp) else np.nan,
        "mean_turning": float(tmp["turning"].mean()) if len(tmp) else np.nan,
        "mean_sinu":    float(tmp["sinu"].mean())    if len(tmp) else np.nan
    }, name=label)

# Always provide All/Flagged; add TP/FN/FP/TN if non-empty
summ_pieces = [
    group_summary(mask_all, "All"),
    group_summary(mask_flag, "Flagged"),
]
if m_tp.any(): summ_pieces.append(group_summary(m_tp, "TP"))
if m_fn.any(): summ_pieces.append(group_summary(m_fn, "FN"))
if m_fp.any(): summ_pieces.append(group_summary(m_fp, "FP"))
if m_tn.any(): summ_pieces.append(group_summary(m_tn, "TN"))

summaries = pd.concat(summ_pieces, axis=1).T

# %diff vs TP for the groups that exist and if TP has rows
if ("TP" in summaries.index) and (summaries.loc["TP", "n_rows"] > 0):
    for col in ["mean_speed","mean_turning","mean_sinu"]:
        base_val = summaries.loc["TP", col]
        for grp in summaries.index:
            if grp == "TP": continue
            summaries.loc[grp, f"{col}_vsTP_%diff"] = (
                (summaries.loc[grp, col] - base_val) / base_val * 100
                if (pd.notna(base_val) and base_val != 0) else np.nan
            )

print(f"\n=== Movement summaries by class ({units_label}) — Holdout ===")
print(summaries.round(3))
summaries.to_csv(outdir / f"ae_holdout_movement_summaries_by_class_{'raw' if units_label=='raw units' else 'scaled'}.csv")

# Direct per-variable summaries
vars_of_interest = [
    "abs_rel_angle", "est", "low", "high",
    "rolling_abs_rel_angle_kurt", "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd", "rolling_abs_rel_angle_skew",
    "rolling_velo_kurt", "rolling_velo_mean", "rolling_velo_sd",
    "rolling_velo_skew", "sinuosity_28pt", "velo"
]
present = [v for v in vars_of_interest if v in df_raw.columns]

def group_summary_vars(mask, label):
    tmp = df_raw.loc[mask, present]
    stats = {f"{v}_mean": tmp[v].mean() for v in present}
    stats.update({f"{v}_sd": tmp[v].std() for v in present})
    stats["n_rows"] = len(tmp)
    return pd.Series(stats, name=label)

summ_vars_pieces = [
    group_summary_vars(mask_all,  "All"),
    group_summary_vars(mask_flag, "Flagged"),
]
if m_tp.any(): summ_vars_pieces.append(group_summary_vars(m_tp, "TP"))
if m_fn.any(): summ_vars_pieces.append(group_summary_vars(m_fn, "FN"))
if m_fp.any(): summ_vars_pieces.append(group_summary_vars(m_fp, "FP"))
if m_tn.any(): summ_vars_pieces.append(group_summary_vars(m_tn, "TN"))

summaries_vars = pd.concat(summ_vars_pieces, axis=1).T

# diffs vs TP (only if TP exists)
if "TP" in summaries_vars.index and summaries_vars.loc["TP","n_rows"] > 0:
    for v in present:
        base = summaries_vars.loc["TP", f"{v}_mean"]
        for grp in summaries_vars.index:
            if grp == "TP": continue
            summaries_vars.loc[grp, f"{v}_diff_vsTP"] = summaries_vars.loc[grp, f"{v}_mean"] - base
            summaries_vars.loc[grp, f"{v}_pctdiff_vsTP"] = (
                (summaries_vars.loc[grp, f"{v}_mean"] - base) / base * 100
                if (pd.notna(base) and base != 0) else np.nan
            )

print("\n=== Per-variable movement summaries — Holdout ===")
print(summaries_vars.round(3))
summaries_vars.to_csv(outdir / f"ae_holdout_movement_summaries_by_variable_{'raw' if units_label=='raw units' else 'scaled'}.csv")

# Residual correlation with movement metrics
movement_links = pd.DataFrame({
    "resid_total": resid_total,
    "speed":       df_raw["speed"],
    "turning":     df_raw["turning"],
    "sinu":        df_raw["sinu"],
}).corr()

print("\n=== Correlation of movement variables with total reconstruction error ===")
print(movement_links.round(3))
movement_links.to_csv(outdir / f"ae_holdout_movement_residual_correlations_{'raw' if units_label=='raw units' else 'scaled'}.csv")

# ----------------------------
# Save the main contribution tables
# ----------------------------
(df_contrib.sort_values("Flagged_%", ascending=False)
          .to_csv(outdir / "ae_holdout_feature_contributions_by_feature.csv", index=False))
cat_roll.to_csv(outdir / "ae_holdout_feature_contributions_by_category.csv")
summary_tbl.to_csv(outdir / "ae_holdout_feature_contrib_summary_simple.csv", index=False)

print(f"\n💾 Saved tables to: {outdir}")
print(f"IDs — TP:{len(TP_ids)}  FN:{len(FN_ids)}  FP:{len(FP_ids)}  TN:{len(TN_ids)}  | Eligible:{len(set(eval_ids))}")


In [ ]:
# --- AE CV-style metrics summary from TP / FP / FN / TN ---

from sklearn.metrics import confusion_matrix

# Define evaluation IDs
ae_eval_ids = sorted((pos_set | neg_set) & set(eligible_idx))
y_true = [1 if i in pos_set else 0 for i in ae_eval_ids]
y_pred = [1 if i in pred_ids else 0 for i in ae_eval_ids]

# Compute confusion matrix
TN, FP, FN, TP = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

# Wilson CI helper
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = z * np.sqrt((p*(1-p)/n) + (z*z/(4*n*n))) / denom
    return (center - half, center + half)

# Compute metrics and CIs
def ae_counts_to_metrics(TN, FP, FN, TP):
    N = TP + TN + FP + FN
    acc  = (TP + TN) / N if N else np.nan
    prec = TP / (TP + FP) if (TP + FP) else np.nan
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    spec = TN / (TN + FP) if (TN + FP) else np.nan
    tss  = sens + spec - 1 if np.isfinite(sens) and np.isfinite(spec) else np.nan

    acc_lo, acc_hi   = wilson_ci(TP + TN, N)
    prec_lo, prec_hi = wilson_ci(TP, TP + FP)
    sens_lo, sens_hi = wilson_ci(TP, TP + FN)
    spec_lo, spec_hi = wilson_ci(TN, TN + FP)
    tss_lo, tss_hi   = (sens_lo + spec_lo - 1, sens_hi + spec_hi - 1) if np.isfinite(sens_lo) and np.isfinite(spec_lo) else (np.nan, np.nan)

    return {
        "mode": "ae",  # Identifier
        "n_eval": N,
        "TP": TP, "FP": FP, "FN": FN, "TN": TN,
        "accuracy": acc, "accuracy_lo": acc_lo, "accuracy_hi": acc_hi,
        "precision": prec, "precision_lo": prec_lo, "precision_hi": prec_hi,
        "tss": tss, "tss_lo": tss_lo, "tss_hi": tss_hi,
    }

ae_metrics = ae_counts_to_metrics(TN, FP, FN, TP)

# Convert to DataFrame
ae_metrics_df = pd.DataFrame([ae_metrics])

# Save to CSV
ae_out_csv = base / "ae_confusion_metrics_summary.csv"
ae_metrics_df.to_csv(ae_out_csv, index=False)
print(f"\n💾 Saved AE confusion metrics → {ae_out_csv}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D  # ⬅️ for clean legend markers

# Use a clean font (optional)
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 11,
    "figure.dpi": 120
})

# ─── Load all summaries into single DataFrame ─────────────────────────────────
models_info = [
    ("60/40", "<LOCAL_PATH>"),
    ("Full Holdout", "<LOCAL_PATH>"),
    ("10%", "<LOCAL_PATH>"),
    ("20%", "<LOCAL_PATH>"),
    ("30%", "<LOCAL_PATH>"),
    ("40%", "<LOCAL_PATH>"),
]

rows = []
for name, path in models_info:
    try:
        df = pd.read_csv(path)
        if "bin" in df.columns:
            df = df[df["bin"].isna()]
        if df.shape[0] > 1:
            df = df.head(1)
        df["model"] = name
        rows.append(df)
    except Exception as e:
        print(f"⚠️ Could not read {name}: {e}")

df_all = pd.concat(rows, ignore_index=True)

# ─── Plotting Function ────────────────────────────────────────────────────────
def plot_cv_comparison_points(df, title="", save_path=None):
    metrics = [
        ("accuracy", "Accuracy", (0, 1.05), 0.1),
        ("precision", "Precision", (0, 1.05), 0.1),
        ("tss", "TSS", (-1.05, 1.05), 0.2),
    ]

    models = df["model"].tolist()
    n_models = len(models)

    # Group definitions
    ae_models = {"60/40", "Full Holdout"}
    cae_models = set(models) - ae_models

    # Two-color palette (consistent across runs)
    AE_COLOR = "#0072B2"   # Blue
    CAE_COLOR = "#E69F00"  # Orange

    # Map each model to its group color
    colors = {m: (AE_COLOR if m in ae_models else CAE_COLOR) for m in models}

    # x-offsets for jittered points
    offsets = np.linspace(-0.3, 0.3, n_models) if n_models > 1 else np.array([0.0])

    fig, axes = plt.subplots(1, 3, figsize=(12.5, 5), sharey=False)

    for ax, (metric, label, ylim, ytick_step) in zip(axes, metrics):
        for x_off, model in zip(offsets, models):
            row = df[df["model"] == model].iloc[0]
            center = row[metric]
            lo = row.get(f"{metric}_lo", np.nan)
            hi = row.get(f"{metric}_hi", np.nan)
            err_lo = 0.0 if np.isnan(lo) else (center - lo)
            err_hi = 0.0 if np.isnan(hi) else (hi - center)

            ax.errorbar(
                x_off, center, yerr=[[err_lo], [err_hi]],
                fmt="o", markersize=7,
                color=colors[model], ecolor="black",
                elinewidth=1.2, capsize=4,
                label="_nolegend_"
            )

        ax.axhline(0, color="black", linewidth=0.6, linestyle="--", alpha=0.4)
        ax.set_xlim(offsets.min() - 0.15, offsets.max() + 0.15)

        # Rotated labels with consistent anchor (fixes "Full Holdout" look)
        ax.set_xticks(offsets)
        ax.set_xticklabels(
            models,
            rotation=35,
            ha="right",
            va="top",
            rotation_mode="anchor"
        )
        ax.tick_params(axis="x", pad=6)

        ax.set_ylim(*ylim)
        ax.set_ylabel(label)
        ax.yaxis.set_major_locator(mticker.MultipleLocator(ytick_step))
        ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
        ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.4)
        ax.set_title("")

    # Title & layout (extra bottom space for x labels + legend)
    plt.suptitle(title, fontsize=15, y=0.98)
    plt.subplots_adjust(bottom=0.30, top=0.88, wspace=0.3)

    # Two-group legend
    proxy_handles = [
        Line2D([0], [0], marker="o", linestyle="None", markersize=7, color=AE_COLOR, label="AE (60/40, Full Holdout)"),
        Line2D([0], [0], marker="o", linestyle="None", markersize=7, color=CAE_COLOR, label="cAE (10%–40%)"),
    ]
    fig.legend(
        handles=proxy_handles,
        loc="lower center",
        ncol=2,
        bbox_to_anchor=(0.5, 0.06),
        frameon=False,
        title="Model Families"
    )

    if save_path:
        fig.savefig(f"{save_path}_points.png", dpi=300, bbox_inches="tight")
        fig.savefig(f"{save_path}_points.pdf", bbox_inches="tight")
        print(f"✅ Saved point plot to {save_path}_points.png / .pdf")

    plt.show()
    plt.close(fig)

# ─── Generate Point Plot ────────────────────────────────────────────
plot_cv_comparison_points(
    df_all,
    save_path="<LOCAL_PATH>"
)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D  # ⬅️ clean legend markers

# Use a clean font (optional)
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titlesize": 26,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "figure.titlesize": 20,
    "figure.dpi": 120
})


# ─── Load all summaries into single DataFrame ─────────────────────────────────
models_info = [
    ("60/40", "<LOCAL_PATH>"),
    ("Full Holdout", "<LOCAL_PATH>"),
    ("10%", "<LOCAL_PATH>"),
    ("20%", "<LOCAL_PATH>"),
    ("30%", "<LOCAL_PATH>"),
    ("40%", "<LOCAL_PATH>"),
]

rows = []
for name, path in models_info:
    try:
        df = pd.read_csv(path)
        if "bin" in df.columns:
            df = df[df["bin"].isna()]
        if df.shape[0] > 1:
            df = df.head(1)
        df["model"] = name
        rows.append(df)
    except Exception as e:
        print(f"⚠️ Could not read {name}: {e}")

df_all = pd.concat(rows, ignore_index=True)

# ─── Plotting Function ────────────────────────────────────────────────────────
def plot_cv_comparison_points(df, title="", save_path=None):
    metrics = [
    ("accuracy", "(a) Accuracy", (0, 1.05), 0.1),
    ("precision", "(b) Precision", (0, 1.05), 0.1),
    ("tss", "(c) TSS", (-1.05, 1.05), 0.2),
]



    models = df["model"].tolist()
    n_models = len(models)

    # Group definitions
    ae_models = {"60/40", "Full Holdout"}
    cae_models = set(models) - ae_models

    # Two-color palette (consistent across runs)
    AE_COLOR = "#0072B2"   # Blue
    CAE_COLOR = "#E69F00"  # Orange

    # Map each model to its group color
    colors = {m: (AE_COLOR if m in ae_models else CAE_COLOR) for m in models}

    # Single marker for all models
    MARKER = "o"

    # x-offsets for jittered points
    offsets = np.linspace(-0.3, 0.3, n_models) if n_models > 1 else np.array([0.0])

    fig, axes = plt.subplots(1, 3, figsize=(12.5, 5), sharey=False)

    for ax, (metric, label, ylim, ytick_step) in zip(axes, metrics):
        for x_off, model in zip(offsets, models):
            row = df[df["model"] == model].iloc[0]
            center = row[metric]
            lo = row.get(f"{metric}_lo", np.nan)
            hi = row.get(f"{metric}_hi", np.nan)
            err_lo = 0.0 if np.isnan(lo) else (center - lo)
            err_hi = 0.0 if np.isnan(hi) else (hi - center)

            ax.errorbar(
                x_off, center, yerr=[[err_lo], [err_hi]],
                fmt=MARKER, linestyle="None", markersize=7,
                markerfacecolor=colors[model], markeredgecolor="black",
                ecolor="black", elinewidth=1.2, capsize=4,
                label="_nolegend_"
            )

        ax.axhline(0, color="black", linewidth=0.6, linestyle="--", alpha=0.4)
        ax.set_xlim(offsets.min() - 0.15, offsets.max() + 0.15)

        # Rotated labels with consistent anchor (fixes "Full Holdout" look)
        ax.set_xticks(offsets)
        ax.set_xticklabels(
            models,
            rotation=35,
            ha="right",
            va="top",
            rotation_mode="anchor"
        )
        ax.tick_params(axis="x", pad=6)

        ax.set_ylim(*ylim)
        ax.set_ylabel(label)
        ax.yaxis.set_major_locator(mticker.MultipleLocator(ytick_step))
        ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
        ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.4)
        ax.set_title("")

    # Title & layout (extra bottom space for x labels + two legends)
    plt.suptitle(title, fontsize=15, y=0.98)
    plt.subplots_adjust(bottom=0.42, top=0.88, wspace=0.45)


    # Legend 1: Model families (colors)
    family_handles = [
        Line2D([0], [0], marker='o', linestyle='None', markersize=7,
               markerfacecolor=AE_COLOR, markeredgecolor='black',
               label="AE (60/40, Full Holdout)"),
        Line2D([0], [0], marker='o', linestyle='None', markersize=7,
               markerfacecolor=CAE_COLOR, markeredgecolor='black',
               label="cAE (10%–40%)"),
    ]
    leg1 = fig.legend(
        handles=family_handles,
        loc="lower center",
        ncol=2,
        bbox_to_anchor=(0.5, 0.08),
        frameon=False,
        title="Model Families"
    )

    fig.add_artist(leg1)

    if save_path:
        fig.savefig(f"{save_path}_points.png", dpi=300, bbox_inches="tight")
        fig.savefig(f"{save_path}_points.pdf", bbox_inches="tight")
        print(f"✅ Saved point plot to {save_path}_points.png / .pdf")

    plt.show()
    plt.close(fig)

# ─── Generate Point Plot ────────────────────────────────────────────
plot_cv_comparison_points(
    df_all,
    save_path="<LOCAL_PATH>"
)


## Feature Importance for each model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "figure.titlesize": 20,
    "figure.dpi": 120
})

def plot_cv_comparison_points_2x2(df, title="", save_path=None):
    # metric_key, y_label, panel_letter, ylim, ytick_step
    metrics = [
        ("accuracy",  "Accuracy",  "A", (0, 1.05), 0.1),
        ("precision", "Precision", "B", (0, 1.05), 0.1),
        ("tss",       "TSS",       "C", (-1.05, 1.05), 0.2),
    ]

    models = df["model"].tolist()
    n_models = len(models)

    ae_models = {"60/40", "Full Holdout"}
    AE_COLOR = "#0072B2"
    CAE_COLOR = "#E69F00"
    colors = {m: (AE_COLOR if m in ae_models else CAE_COLOR) for m in models}

    marker_cycle = ['o', 's', '^', 'D', 'v', 'P']
    markers = {m: marker_cycle[i % len(marker_cycle)] for i, m in enumerate(models)}

    offsets = np.linspace(-0.3, 0.3, n_models) if n_models > 1 else np.array([0.0])

    # --- 2x2 layout: A B / C Legend ---
    fig, axs = plt.subplots(2, 2, figsize=(12.5, 8.2))
    axA, axB = axs[0, 0], axs[0, 1]
    axC, axL = axs[1, 0], axs[1, 1]   # axL = legend panel
    plot_axes = [axA, axB, axC]

    for ax, (metric, ylabel, letter, ylim, ytick_step) in zip(plot_axes, metrics):
        # points + errorbars
        for x_off, model in zip(offsets, models):
            row = df[df["model"] == model].iloc[0]
            center = row[metric]
            lo = row.get(f"{metric}_lo", np.nan)
            hi = row.get(f"{metric}_hi", np.nan)
            err_lo = 0.0 if np.isnan(lo) else (center - lo)
            err_hi = 0.0 if np.isnan(hi) else (hi - center)

            ax.errorbar(
                x_off, center, yerr=[[err_lo], [err_hi]],
                fmt=markers[model],
                linestyle="None",
                markersize=7,
                markerfacecolor=colors[model],
                markeredgecolor="black",
                ecolor="black",
                elinewidth=1.2,
                capsize=4
            )

        ax.axhline(0, color="black", linewidth=0.6, linestyle="--", alpha=0.4)
        ax.set_xlim(offsets.min() - 0.15, offsets.max() + 0.15)

        ax.set_xticks(offsets)
        ax.set_xticklabels(models, rotation=35, ha="right", va="top", rotation_mode="anchor")
        ax.tick_params(axis="x", pad=6)

        ax.set_ylim(*ylim)
        ax.set_ylabel(ylabel)
        ax.yaxis.set_major_locator(mticker.MultipleLocator(ytick_step))
        ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
        ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.4)

        # Panel letter in top-left (classic style). Move to top-right if you prefer.
        ax.text(
            -0.30, 1.05, letter,                  # x<0 and y>1 pushes it outside
            transform=ax.transAxes,
            ha="left", va="bottom",
            fontsize=18, fontweight="bold",
            clip_on=False
        )

    # --- Legend panel (bottom-right) ---
    axL.axis("off")

    # Legend 1: Model families (colors)
    family_handles = [
        Line2D([0], [0], marker='o', linestyle='None', markersize=9,
               markerfacecolor=AE_COLOR, markeredgecolor='black',
               label="AE (60/40, Full Holdout)"),
        Line2D([0], [0], marker='o', linestyle='None', markersize=9,
               markerfacecolor=CAE_COLOR, markeredgecolor='black',
               label="cAE (10%–40%)"),
    ]
    leg1 = axL.legend(
        handles=family_handles,
        loc="upper left",
        frameon=False,
        title="Model Families"
    )
    axL.add_artist(leg1)

    # Legend 2: Individual models (shapes only)
    shape_handles = [
        Line2D([0], [0], marker=markers[m], linestyle='None', markersize=9,
               markerfacecolor='white', markeredgecolor='black', label=m)
        for m in models
    ]
    axL.legend(
        handles=shape_handles,
        loc="upper left",
        bbox_to_anchor=(0.0, 0.55),
        frameon=False,
        title="Models"
    )

    fig.suptitle(title, y=0.98)

    # spacing tuned for rotated x labels
    plt.subplots_adjust(wspace=0.35, hspace=0.55, top=0.92)

    if save_path:
        fig.savefig(f"{save_path}_points_2x2.png", dpi=300, bbox_inches="tight")
        fig.savefig(f"{save_path}_points_2x2.pdf", bbox_inches="tight")
        print(f"✅ Saved to {save_path}_points_2x2.png / .pdf")

    plt.show()
    plt.close(fig)


# call it
plot_cv_comparison_points_2x2(
    df_all,
    title="",
    save_path="<LOCAL_PATH>"
)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from pathlib import Path
from matplotlib.offsetbox import TextArea, VPacker, AnchoredOffsetbox


# ─── Config ─────────────────────────────────────────────
MODEL_ORDER = ["60/40", "Full Holdout", "10%", "20%", "30%", "40%"]
CLASS_ORDER = ["FP", "FN"]
USE_NUMERIC_Y = True

CLASS_COLORS = {"FP": "#E69F00", "FN": "#0072B2"}

SAVE_BASE = Path("<LOCAL_PATH>")
SAVE_BASE.mkdir(parents=True, exist_ok=True)

files = {
    "60/40": {
        "FP": "<LOCAL_PATH>",
        "FN": "<LOCAL_PATH>",
    },
    "Full Holdout": {
        "FP": "<LOCAL_PATH>",
        "FN": "<LOCAL_PATH>",
    },
    "10%": {
        "FP": "<LOCAL_PATH>",
        "FN": "<LOCAL_PATH>",
    },
    "20%": {
        "FP": "<LOCAL_PATH>",
        "FN": "<LOCAL_PATH>",
    },
    "30%": {
        "FP": "<LOCAL_PATH>",
        "FN": "<LOCAL_PATH>",
    },
    "40%": {
        "FP": "<LOCAL_PATH>",
        "FN": "<LOCAL_PATH>",
    },
}

pretty_order = [
    "velo", "rolling_velo_mean", "rolling_velo_sd",
    "rolling_velo_skew", "rolling_velo_kurt",
    "abs_rel_angle", "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd", "rolling_abs_rel_angle_skew",
    "rolling_abs_rel_angle_kurt",
    "sinuosity_28pt",
    "est", "low", "high"
]

pretty_map = {
    "velo": "Velocity",
    "rolling_velo_mean": "Rolling Velocity Mean",
    "rolling_velo_sd": "Rolling Velocity SD",
    "rolling_velo_skew": "Rolling Velocity Skew",
    "rolling_velo_kurt": "Rolling Velocity Kurtosis",
    "abs_rel_angle": "Absolute Relative Angle",
    "rolling_abs_rel_angle_mean": "Rolling Abs Rel Angle Mean",
    "rolling_abs_rel_angle_sd": "Rolling Abs Rel Angle SD",
    "rolling_abs_rel_angle_skew": "Rolling Abs Rel Angle Skew",
    "rolling_abs_rel_angle_kurt": "Rolling Abs Rel Angle Kurtosis",
    "sinuosity_28pt": "Sinuosity",
    "est": "Median Space Usage",
    "low": "Low CI Space Usage",
    "high": "High CI Space Usage",
}

def load_one(path, class_label):
    df = pd.read_csv(path)
    delta_col = next((c for c in df.columns if c.startswith("Delta_") and c.endswith("_minus_All_pp")), None)
    if delta_col is None:
        if f"{class_label}_%" in df.columns and "All_%" in df.columns:
            df["Delta_pp"] = df[f"{class_label}_%"] - df["All_%"]
        else:
            raise KeyError(f"No delta column found in {path}")
    else:
        df["Delta_pp"] = df[delta_col]
    return df[["feature", "Delta_pp"]].copy()

MARKER = "o"   # single marker for all models

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "figure.dpi": 160,
})

# ─── Build combined long_df ─────────────────────────────
long_dfs = []
for model, class_dict in files.items():
    for cls, path in class_dict.items():
        df = load_one(path, cls)
        df["model"] = model
        df["class"] = cls

        missing = set(pretty_order) - set(df["feature"])
        if missing:
            df_missing = pd.DataFrame({
                "feature": list(missing),
                "Delta_pp": 0.0,
                "model": model,
                "class": cls
            })
            df = pd.concat([df, df_missing], ignore_index=True)

        long_dfs.append(df)

long_df = pd.concat(long_dfs, ignore_index=True)
long_df = long_df[long_df["feature"].isin(pretty_order)].copy()

feat_order = [f for f in pretty_order if f in long_df["feature"].unique()]
y_pos = {f: i for i, f in enumerate(feat_order[::-1])}
y_ticks = [y_pos[f] for f in feat_order]

feat_nums = {f: i + 1 for i, f in enumerate(feat_order)}
if USE_NUMERIC_Y:
    y_ticklabels = [str(feat_nums[f]) for f in feat_order]
    mapping_lines = [f"{feat_nums[f]} = {pretty_map.get(f, f)}" for f in feat_order]
else:
    y_ticklabels = [pretty_map.get(f, f) for f in feat_order]
    mapping_lines = None

def plot_model_subset(models_subset, panel_letters, out_stub, feature_key_y=0.58):
    n = len(models_subset)

    fig_w = 13.8 if USE_NUMERIC_Y else 12.0
    fig_h = 3.8 * n  # keep your current “taller” panels

    fig = plt.figure(figsize=(fig_w, fig_h), constrained_layout=True)
    gs = fig.add_gridspec(nrows=n, ncols=2, width_ratios=[5.2, 2.0])

    axes = [fig.add_subplot(gs[i, 0]) for i in range(n)]
    key_ax = fig.add_subplot(gs[:, 1])
    key_ax.axis("off")

    for i in range(1, n):
        axes[i].sharey(axes[0])

    for i, (ax, model) in enumerate(zip(axes, models_subset)):
        sub_df = long_df[long_df["model"] == model]

        ax.axvline(0, color="black", lw=0.9, ls="--", alpha=0.6)
        ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.5)

        for c in CLASS_ORDER:
            sub = sub_df[sub_df["class"] == c]
            y = [y_pos[f] for f in sub["feature"]]
            x = sub["Delta_pp"].to_numpy()
            ax.scatter(
                x, y,
                s=85,
                marker=MARKER,
                facecolor=CLASS_COLORS[c],
                edgecolor="black",
                linewidth=0.8,
                zorder=3
            )

        # panel label outside (left/top)
        ax.text(
            -0.10, 1.03,
            f"({panel_letters[i]})",
            transform=ax.transAxes,
            ha="left", va="bottom",
            fontsize=16, fontweight="bold",
            clip_on=False
        )

        ax.set_title(model, loc="left", pad=6, fontsize=15)
        ax.set_xlabel("Δ (percentage points)")
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(y_ticklabels)
        ax.tick_params(axis="y", labelsize=15)

    axes[len(axes)//2].set_ylabel("Feature (see key)" if USE_NUMERIC_Y else "Feature", fontsize=16)

    # Legends (right panel)
    color_handles = [
        Line2D([0], [0], marker='o', linestyle="None",
               markersize=9, markerfacecolor=CLASS_COLORS[c], markeredgecolor="black", label=c)
        for c in CLASS_ORDER
    ]

    leg1 = key_ax.legend(
        handles=color_handles, title="Class",
        loc="upper left", bbox_to_anchor=(0.0, 1.0),
        frameon=False, borderaxespad=0.0
    )
    key_ax.add_artist(leg1)

    # ✅ Feature key: deterministic spacing between title and list
    # ✅ Feature key using OffsetBox so title-to-list spacing is fixed in POINTS
    if USE_NUMERIC_Y:
        # Build title + body as separate text areas
        title_ta = TextArea(
            "Feature key",
            textprops=dict(size=13, weight="bold", family="DejaVu Sans")
        )

        body_ta = TextArea(
            "\n".join(mapping_lines),
            textprops=dict(size=13, family="DejaVu Sans", linespacing=1.05)
        )

        # Pack vertically with a FIXED gap (sep) in points
        packed = VPacker(children=[title_ta, body_ta], align="left", pad=0, sep=6)

        # Anchor the packed block inside key_ax
        anchored_box = AnchoredOffsetbox(
            loc="upper left",
            child=packed,
            pad=0.0,
            borderpad=0.0,
            frameon=False,
            bbox_to_anchor=(0.0, feature_key_y),    # anchor point in axes fraction
            bbox_transform=key_ax.transAxes
        )

        key_ax.add_artist(anchored_box)



    save_path = SAVE_BASE / out_stub
    fig.savefig(f"{save_path}.png", dpi=300)
    fig.savefig(f"{save_path}.pdf", dpi=300)
    print(f"✅ Saved {save_path}.png / .pdf")

    plt.show()
    plt.close(fig)

# ─── Split you want ─────────────────────────────────────

# Figure 3A (AE-only): move feature key UP to match the other figure visually
plot_model_subset(
    models_subset=["60/40", "Full Holdout"],
    panel_letters=["a", "b"],
    out_stub="Figure3A_Deltas_AE_only_60-40_vs_FullHoldout",
    feature_key_y=0.40   # <-- tweak (0.66–0.74) to taste
)

# Figure 3B (cAE 10–40): keep as-is
plot_model_subset(
    models_subset=["10%", "20%", "30%", "40%"],
    panel_letters=["c", "d", "e", "f"],
    out_stub="Figure3B_Deltas_cAE_10_to_40",
    feature_key_y=0.62
)


In [ ]:

def get_range(feature, cls, models):
    sub = long_df[
        (long_df["feature"] == feature) &
        (long_df["class"] == cls) &
        (long_df["model"].isin(models))
    ]
    return round(sub["Delta_pp"].min(), 1), round(sub["Delta_pp"].max(), 1)

# Example:
print("FN rolling_abs_rel_angle_mean:", get_range("rolling_abs_rel_angle_mean", "FN", ["60/40", "Full Holdout"]))
print("FN rolling_abs_rel_angle_skew:", get_range("rolling_abs_rel_angle_skew", "FN", ["60/40", "Full Holdout"]))
print("FN rolling_velo_sd:", get_range("rolling_velo_sd", "FN", ["60/40", "Full Holdout"]))

models_cae = ["10%", "20%", "30%", "40%"]

# FN - space use (low)
print("FN low:", get_range("low", "FN", ["10%", "20%", "30%"]))

# Model-specific FN
print("10% FN angle mean:", get_range("rolling_abs_rel_angle_mean", "FN", ["10%"]))
print("20% FN angle sd:", get_range("rolling_abs_rel_angle_sd", "FN", ["20%"]))
print("40% FN velo mean:", get_range("rolling_velo_mean", "FN", ["40%"]))

# FP equivalents
print("FP low:", get_range("low", "FP", ["10%", "20%"]))
print("10% FP velo mean:", get_range("rolling_velo_mean", "FP", ["10%"]))
print("10% FP velo sd:", get_range("rolling_velo_sd", "FP", ["10%"]))
print("20% FP angle sd:", get_range("rolling_abs_rel_angle_sd", "FP", ["20%"]))


In [ ]:
# =============================================================================
# Confirm reported Δ values from long_df
# =============================================================================

def get_delta_table(feature, cls, models):
    """
    Return model-specific Delta_pp values for a feature/class/model subset.
    This is better than only printing min/max because it shows exactly
    which model and feature number produced each value.
    """
    sub = long_df[
        (long_df["feature"] == feature) &
        (long_df["class"] == cls) &
        (long_df["model"].isin(models))
    ].copy()

    if sub.empty:
        print(f"⚠️ No rows found for feature={feature}, class={cls}, models={models}")
        return pd.DataFrame()

    sub["feature_num"] = sub["feature"].map(feat_nums)
    sub["pretty_feature"] = sub["feature"].map(pretty_map)
    sub["Delta_pp_rounded"] = sub["Delta_pp"].round(1)

    sub = sub[[
        "model",
        "class",
        "feature_num",
        "feature",
        "pretty_feature",
        "Delta_pp",
        "Delta_pp_rounded"
    ]].sort_values(["model", "class", "feature_num"])

    return sub


def print_range_and_values(label, feature, cls, models):
    """
    Print both the range and the model-specific values.
    """
    sub = get_delta_table(feature, cls, models)

    if sub.empty:
        return

    min_val = round(sub["Delta_pp"].min(), 1)
    max_val = round(sub["Delta_pp"].max(), 1)

    print("\n" + "=" * 90)
    print(label)
    print(f"Feature: {feature} | {pretty_map.get(feature, feature)}")
    print(f"Class: {cls}")
    print(f"Models: {models}")
    print(f"Range: {min_val} to {max_val} percentage points")
    print("-" * 90)
    print(sub.to_string(index=False))


# =============================================================================
# AE-only values: Figure 3A
# =============================================================================

models_ae = ["60/40", "Full Holdout"]

print_range_and_values(
    label="AE-only FN rolling absolute relative angle mean",
    feature="rolling_abs_rel_angle_mean",
    cls="FN",
    models=models_ae
)

print_range_and_values(
    label="AE-only FN rolling absolute relative angle skew",
    feature="rolling_abs_rel_angle_skew",
    cls="FN",
    models=models_ae
)

print_range_and_values(
    label="AE-only FN rolling velocity SD",
    feature="rolling_velo_sd",
    cls="FN",
    models=models_ae
)


# =============================================================================
# cAE values: Figure 3B / Figure 4 wording checks
# =============================================================================

models_cae = ["10%", "20%", "30%", "40%"]

# FN - space use lower bound
print_range_and_values(
    label="cAE FN low CI space usage",
    feature="low",
    cls="FN",
    models=["10%", "20%", "30%"]
)

# Model-specific FN checks
print_range_and_values(
    label="10% FN rolling absolute relative angle mean",
    feature="rolling_abs_rel_angle_mean",
    cls="FN",
    models=["10%"]
)

print_range_and_values(
    label="20% FN rolling absolute relative angle SD",
    feature="rolling_abs_rel_angle_sd",
    cls="FN",
    models=["20%"]
)

print_range_and_values(
    label="40% FN rolling velocity mean",
    feature="rolling_velo_mean",
    cls="FN",
    models=["40%"]
)

print_range_and_values(
    label="40% FN rolling velocity SD",
    feature="rolling_velo_sd",
    cls="FN",
    models=["40%"]
)


# =============================================================================
# FP equivalents
# =============================================================================

print_range_and_values(
    label="cAE FP low CI space usage",
    feature="low",
    cls="FP",
    models=["10%", "20%"]
)

print_range_and_values(
    label="10% FP rolling velocity mean",
    feature="rolling_velo_mean",
    cls="FP",
    models=["10%"]
)

print_range_and_values(
    label="10% FP rolling velocity SD",
    feature="rolling_velo_sd",
    cls="FP",
    models=["10%"]
)

print_range_and_values(
    label="20% FP rolling absolute relative angle SD",
    feature="rolling_abs_rel_angle_sd",
    cls="FP",
    models=["20%"]
)

In [ ]:
# =============================================================================
# Confirm Figure 4 value:
# 40% model, false negatives, rolling velocity standard deviation
# =============================================================================

target = long_df[
    (long_df["model"] == "40%") &
    (long_df["class"] == "FN") &
    (long_df["feature"] == "rolling_velo_sd")
].copy()

target["feature_num"] = target["feature"].map(feat_nums)
target["pretty_feature"] = target["feature"].map(pretty_map)
target["Delta_pp_rounded"] = target["Delta_pp"].round(1)

print(target[
    ["model", "class", "feature_num", "feature", "pretty_feature", "Delta_pp", "Delta_pp_rounded"]
].to_string(index=False))

In [ ]:
compare_40_fn = long_df[
    (long_df["model"] == "40%") &
    (long_df["class"] == "FN") &
    (long_df["feature"].isin(["rolling_velo_mean", "rolling_velo_sd"]))
].copy()

compare_40_fn["feature_num"] = compare_40_fn["feature"].map(feat_nums)
compare_40_fn["pretty_feature"] = compare_40_fn["feature"].map(pretty_map)
compare_40_fn["Delta_pp_rounded"] = compare_40_fn["Delta_pp"].round(1)

print(compare_40_fn[
    ["model", "class", "feature_num", "feature", "pretty_feature", "Delta_pp", "Delta_pp_rounded"]
].sort_values("feature_num").to_string(index=False))